In [ ]:
import pandas as pd
from pathlib import Path

In [ ]:
PROCESSED_DIR = Path("../data/processed")

historical_file = (
    PROCESSED_DIR
    / "historical_matches_v2.csv"
)

historical_df = pd.read_csv(
    historical_file
)

historical_df["date"] = pd.to_datetime(
    historical_df["date"],
    utc=True,
    errors="coerce"
)

print(
    "Historical shape:",
    historical_df.shape
)

print(
    "Date range:",
    historical_df["date"].min(),
    "→",
    historical_df["date"].max()
)

In [ ]:
# =========================================================
# V2 INPUT AUDIT
# =========================================================

pl = historical_df[
    historical_df["competition"] == "premier_league"
].copy()

print("PL matches loaded:", len(pl))

print("\nMatches by season:")
print(
    pl["season"]
    .value_counts()
    .sort_index()
)

print("\nDate range:")
print(
    pl["date"].min(),
    "→",
    pl["date"].max()
)

print("\nMissing values by season:")

check_cols = [
    "home_goals",
    "away_goals",
    "home_xg",
    "away_xg",
    "home_shots",
    "away_shots",
    "home_shots_on_target",
    "away_shots_on_target",
    "home_possession",
    "away_possession",
    "home_passes",
    "away_passes",
    "home_corners",
    "away_corners",
]

missing = (
    pl.groupby("season")[check_cols]
    .agg(lambda x: x.isna().sum())
)

print(missing.to_string())

In [ ]:
from pathlib import Path
import pandas as pd

PROCESSED_DIR = Path("../data/processed")

files_to_audit = [
    "historical_matches.csv",
    "historical_matches_v2.csv",
    "pl_match_features.csv",
    "model_features.csv",
    "elo_performance_features.csv",
    "opponent_strength_features.csv",
    "dominance_features.csv",
    "european_performance_features.csv",
]

for filename in files_to_audit:
    
    path = PROCESSED_DIR / filename
    
    print("\n" + "=" * 70)
    print(filename)
    print("=" * 70)
    
    if not path.exists():
        print("FILE NOT FOUND")
        continue
    
    df = pd.read_csv(path)
    
    print("Shape:", df.shape)
    
    print("Columns:")
    print(df.columns.tolist())
    
    # Date information
    date_cols = [
        c for c in df.columns
        if "date" in c.lower()
    ]
    
    if date_cols:
        for col in date_cols:
            dates = pd.to_datetime(
                df[col],
                errors="coerce",
                utc=True
            )
            
            print(
                f"{col}:",
                dates.min(),
                "→",
                dates.max()
            )
    
    # Season information
    if "season" in df.columns:
        print("Seasons:")
        print(
            df["season"]
            .value_counts()
            .sort_index()
            .to_dict()
        )

In [ ]:
files_to_audit = [
    "pl_match_features.csv",
    "model_features.csv",
    "elo_performance_features.csv",
    "opponent_strength_features.csv",
    "dominance_features.csv",
    "european_performance_features.csv",
]

for filename in files_to_audit:

    path = PROCESSED_DIR / filename
    df = pd.read_csv(path)

    print(
        filename,
        "| rows:", len(df),
        "| cols:", len(df.columns),
        "| match_id unique:",
        df["match_id"].nunique() if "match_id" in df.columns else "N/A"
    )

In [ ]:
for filename in [
    "pl_match_features.csv",
    "model_features.csv"
]:

    df = pd.read_csv(PROCESSED_DIR / filename)

    print("\n", filename.upper())

    if "season" in df.columns:
        print(
            df["season"]
            .value_counts()
            .sort_index()
        )

    print("\nColumns:")
    for col in df.columns:
        print(col)

In [ ]:
model_features = pd.read_csv(
    PROCESSED_DIR / "model_features.csv"
)

print("Shape:", model_features.shape)

print("\nSeasons:")
print(
    model_features["season"]
    .value_counts()
    .sort_index()
)

print("\nDate range:")
model_features["date"] = pd.to_datetime(
    model_features["date"],
    utc=True,
    errors="coerce"
)

print(
    model_features["date"].min(),
    "→",
    model_features["date"].max()
)

In [ ]:
print("Columns with missing values:")

missing = (
    model_features
    .isna()
    .sum()
)

print(
    missing[missing > 0]
    .sort_values(ascending=False)
    .to_string()
)

In [ ]:
import json
import pprint
from pathlib import Path

RAW_MATCH_DIR = Path("../data/raw/matches")

sample_file = list(
    (RAW_MATCH_DIR / "premier_league" / "2023_24").glob("*.json")
)[0]

with open(sample_file, "r", encoding="utf-8") as f:
    sample = json.load(f)

details = sample["details"]

if isinstance(details, dict):
    details = details.get("data", details)

print("DETAIL KEYS:")
print(details.keys())

print("\nFULL DETAILS:")
pprint.pp(details, depth=5)

In [ ]:
# Date conversion
historical_df["date"] = pd.to_datetime(
    historical_df["date"],
    utc = True
)

historical_df.dtypes

In [ ]:
# For feature engineering purposes, we want to make home-team rows (2,280 rows)
# Makes it easier to calculate rolling averages since for example Arsenal will always be in the same team column.
# All the stats will be written from Arsenal's perspective.
home_rows = historical_df[
    [
        "match_id",
        "date",
        "season",
        "matchday",
        "home_team",
        "away_team",
        "home_goals",
        "away_goals",
        "home_xg",
        "away_xg",
        "home_shots",
        "away_shots",
        "home_shots_on_target",
        "away_shots_on_target",
        "home_possession",
        "away_possession",
        "home_passes",
        "away_passes",
        "home_corners",
        "away_corners"
    ]
].copy()

In [ ]:
home_rows.shape

In [ ]:
# For every row the home team will become "team" and the away team will become "opponent".
# Stats will become "for" and "against".

home_rows = home_rows.rename(columns = {
    "home_team": "team",
    "away_team": "opponent",

    "home_goals": "goals_for",
    "away_goals": "goals_against",

    "home_xg": "xg_for",
    "away_xg": "xg_against",

    "home_shots": "shots_for",
    "away_shots": "shots_against",

    "home_shots_on_target": "shots_on_target_for",
    "away_shots_on_target": "shots_on_target_against",

    "home_possession": "possession_for",
    "away_possession": "possession_against",

    "home_passes": "passes_for",
    "away_passes": "passes_against",

    "home_corners": "corners_for",
    "away_corners": "corners_against"
})

home_rows["venue"] = "home"

In [ ]:
home_rows.head()

In [ ]:
# Do the same for away-team now
away_rows = historical_df[
    [
        "match_id",
        "date",
        "season",
        "matchday",
        "home_team",
        "away_team",
        "home_goals",
        "away_goals",
        "home_xg",
        "away_xg",
        "home_shots",
        "away_shots",
        "home_shots_on_target",
        "away_shots_on_target",
        "home_possession",
        "away_possession",
        "home_passes",
        "away_passes",
        "home_corners",
        "away_corners"
    ]
].copy()

In [ ]:
away_rows = away_rows.rename(columns={
    "away_team": "team",
    "home_team": "opponent",

    "away_goals": "goals_for",
    "home_goals": "goals_against",

    "away_xg": "xg_for",
    "home_xg": "xg_against",

    "away_shots": "shots_for",
    "home_shots": "shots_against",

    "away_shots_on_target": "shots_on_target_for",
    "home_shots_on_target": "shots_on_target_against",

    "away_possession": "possession_for",
    "home_possession": "possession_against",

    "away_passes": "passes_for",
    "home_passes": "passes_against",

    "away_corners": "corners_for",
    "home_corners": "corners_against"
})

away_rows["venue"] = "away"

In [ ]:
away_rows.head()

In [ ]:
# Combine home rows and away rows

team_columns = [
    "match_id",
    "date",
    "season",
    "matchday",
    "team",
    "opponent",
    "venue",
    "goals_for",
    "goals_against",
    "xg_for",
    "xg_against",
    "shots_for",
    "shots_against",
    "shots_on_target_for",
    "shots_on_target_against",
    "possession_for",
    "possession_against",
    "passes_for",
    "passes_against",
    "corners_for",
    "corners_against"
]

home_rows = home_rows[team_columns]
away_rows = away_rows[team_columns]

In [ ]:
# Combine home and away rows

team_matches = pd.concat(
    [home_rows, away_rows],
    ignore_index = True
)

team_matches.shape

In [ ]:
# Sort each team's matches chronologically

team_matches = team_matches.sort_values(
    ["team", "date"]
).reset_index(drop = True)

In [ ]:
# Inspecting Arsenal's matches
team_matches[
    team_matches["team"] == "Arsenal"
][
    [
        "date",
        "team",
        "opponent",
        "venue",
        "goals_for",
        "goals_against"
    ]
].head(10)

In [ ]:
# Looking for each team's average goals scored over previous 5 matches
# Creating a pre-match rolling feature.

team_matches["avg_goals_for_last5"] = (
    team_matches

    # Calculating separately for each team.
    .groupby("team")["goals_for"]
    .transform(
        # Moving the data down one match (shift(1)) so the current match is excluded.
        # shift(1) prevents data leakage.
        # Averaging the previous 5 matches (rolling(5))
        lambda x: x.shift(1).rolling(5).mean()
    )
)

In [ ]:
# Inspecting Arsenal again

team_matches[
    team_matches["team"] == "Arsenal"
][
    [
        "date",
        "opponent",
        "goals_for",
        "avg_goals_for_last5"
    ]
].head(10)

In [ ]:
# Applying same rolling logic to the rest of the main team stats

rolling_stats = [
    "goals_for",
    "goals_against",
    "xg_for",
    "xg_against",
    "shots_for",
    "shots_against",
    "shots_on_target_for",
    "shots_on_target_against",
    "possession_for",
    "possession_against",
    "passes_for",
    "passes_against",
    "corners_for",
    "corners_against"
]

for stat in rolling_stats:
    team_matches[f"avg_{stat}_last5"] = (
        team_matches
        .groupby("team")[stat]
        .transform(
            lambda x: x.shift(1).rolling(5).mean()
        )
    )

In [ ]:
team_matches[
    team_matches["team"] == "Arsenal"
][
    [
        "date",
        "opponent",
        "avg_goals_for_last5",
        "avg_goals_against_last5",
        "avg_xg_for_last5",
        "avg_xg_against_last5",
        "avg_shots_for_last5"
    ]
].head(10)

In [ ]:
# Converting results into league points.

team_matches["points"] = 0

team_matches.loc[
    team_matches["goals_for"] > team_matches["goals_against"],
    "points"
] = 3

team_matches.loc[
    team_matches["goals_for"] == team_matches["goals_against"],
    "points"
] = 1

In [ ]:
# Creating a leakage-safe rolling total over previous 5 matches.

team_matches["points_last5"] = (
    team_matches
    .groupby("team")["points"]
    .transform(
        lambda x: x.shift(1).rolling(5).sum()
    )
)

In [ ]:
team_matches[
    team_matches["team"] == "Arsenal"
][
    [
        "date",
        "opponent",
        "goals_for",
        "goals_against",
        "points",
        "points_last5"
    ]
].head(10)

In [ ]:
# Create a W/D/L column

team_matches["result"] = "D"

team_matches.loc[
    team_matches["goals_for"] > team_matches["goals_against"],
    "result"
] = "W"

team_matches.loc[
    team_matches["goals_for"] < team_matches["goals_against"],
    "result"
] = "L"

In [ ]:
team_matches[
    team_matches["team"] == "Arsenal"
][
    [
        "date",
        "opponent",
        "venue",
        "goals_for",
        "goals_against",
        "result",
        "points_last5"
    ]
].head(10)

In [ ]:
# Creating a rolling goal difference feature over previous 5 matches.

team_matches["goal_difference"] = (
    team_matches["goals_for"] - team_matches["goals_against"]
)

In [ ]:
team_matches["goal_difference_last5"] = (
    team_matches
    .groupby("team")["goal_difference"]
    .transform(
        lambda x: x.shift(1).rolling(5).sum()
    )
)

In [ ]:
team_matches[
    team_matches["team"] == "Arsenal"
][
    [
        "date",
        "opponent",
        "result",
        "points_last5",
        "goal_difference_last5"
    ]
].head(10)

In [ ]:
# Adding a days of rest feature
# Useful because teams with few days between matches may be more fatigued.

team_matches["days_rest"] = (
    team_matches
    .groupby("team")["date"]
    .diff()
    .dt.total_seconds()
    / 86400
)

In [ ]:
team_matches[
    team_matches["team"] == "Arsenal"
][
    [
        "date",
        "opponent",
        "venue",
        "days_rest"
    ]
].head(10)

In [ ]:
# Rounding days of rest to look cleaner.

team_matches["days_rest"] = team_matches["days_rest"].round(1)

In [ ]:
# Inspecting how a team performs home and away

team_matches["points_last3_same_venue"] = (
    team_matches
    # Means previous 3 matches at the same venue type
    .groupby(["team", "venue"])["points"]
    .transform(
        lambda x: x.shift(1).rolling(3).sum()
    )
)

In [ ]:
team_matches[
    team_matches["team"] == "Arsenal"
][
    [
        "date",
        "opponent",
        "venue",
        "points",
        "points_last5",
        "points_last3_same_venue"
    ]
].head(15)

In [ ]:
# Identifying our feature columns that we want to include.

feature_columns = [
    "match_id",
    "team",
    "avg_goals_for_last5",
    "avg_goals_against_last5",
    "avg_xg_for_last5",
    "avg_xg_against_last5",
    "avg_shots_for_last5",
    "avg_shots_against_last5",
    "avg_shots_on_target_for_last5",
    "avg_shots_on_target_against_last5",
    "avg_possession_for_last5",
    "avg_possession_against_last5",
    "avg_passes_for_last5",
    "avg_passes_against_last5",
    "avg_corners_for_last5",
    "avg_corners_against_last5",
    "points_last5",
    "points_last10",
    "win_rate_last5",
    "goal_difference_last5",
    "xg_difference_last5",
    "points_last3_same_venue",
    "days_rest"
]

In [ ]:
team_features = team_matches[feature_columns].copy()

team_features.shape

In [ ]:
# Split these team-feature tows into home-team features and away-team features.

home_features = team_features[
    team_matches["venue"] == "home"
].copy()

away_features = team_features[
    team_matches["venue"] == "away"
].copy()

In [ ]:
# Rename the columns so they're labeled for the final match-level dataset

home_features = home_features.rename(columns={
    "team": "home_team",
    "avg_goals_for_last5": "home_avg_goals_for_last5",
    "avg_goals_against_last5": "home_avg_goals_against_last5",
    "avg_xg_for_last5": "home_avg_xg_for_last5",
    "avg_xg_against_last5": "home_avg_xg_against_last5",
    "avg_shots_for_last5": "home_avg_shots_for_last5",
    "avg_shots_against_last5": "home_avg_shots_against_last5",
    "avg_shots_on_target_for_last5": "home_avg_shots_on_target_for_last5",
    "avg_shots_on_target_against_last5": "home_avg_shots_on_target_against_last5",
    "avg_possession_for_last5": "home_avg_possession_for_last5",
    "avg_possession_against_last5": "home_avg_possession_against_last5",
    "avg_passes_for_last5": "home_avg_passes_for_last5",
    "avg_passes_against_last5": "home_avg_passes_against_last5",
    "avg_corners_for_last5": "home_avg_corners_for_last5",
    "avg_corners_against_last5": "home_avg_corners_against_last5",
    "points_last5": "home_points_last5",
    "points_last10": "home_points_last10",
    "win_rate_last5": "home_win_rate_last5",
    "goal_difference_last5": "home_goal_difference_last5",
    "xg_difference_last5": "home_xg_difference_last5",
    "points_last3_same_venue": "home_points_last3_same_venue",
    "days_rest": "home_days_rest"
})

In [ ]:
away_features = away_features.rename(columns={
    "team": "away_team",
    "avg_goals_for_last5": "away_avg_goals_for_last5",
    "avg_goals_against_last5": "away_avg_goals_against_last5",
    "avg_xg_for_last5": "away_avg_xg_for_last5",
    "avg_xg_against_last5": "away_avg_xg_against_last5",
    "avg_shots_for_last5": "away_avg_shots_for_last5",
    "avg_shots_against_last5": "away_avg_shots_against_last5",
    "avg_shots_on_target_for_last5": "away_avg_shots_on_target_for_last5",
    "avg_shots_on_target_against_last5": "away_avg_shots_on_target_against_last5",
    "avg_possession_for_last5": "away_avg_possession_for_last5",
    "avg_possession_against_last5": "away_avg_possession_against_last5",
    "avg_passes_for_last5": "away_avg_passes_for_last5",
    "avg_passes_against_last5": "away_avg_passes_against_last5",
    "avg_corners_for_last5": "away_avg_corners_for_last5",
    "avg_corners_against_last5": "away_avg_corners_against_last5",
    "points_last5": "away_points_last5",
    "points_last10": "away_points_last10",
    "win_rate_last5": "away_win_rate_last5",
    "goal_difference_last5": "away_goal_difference_last5",
    "xg_difference_last5": "away_xg_difference_last5",
    "points_last3_same_venue": "away_points_last3_same_venue",
    "days_rest": "away_days_rest"
})

In [ ]:
print(home_features.shape)
print(away_features.shape)

In [ ]:
# Merge both sides back onto the original match table.

model_df = historical_df.merge(
    home_features,
    on = ["match_id", "home_team"],
    how = "left"
)

model_df = model_df.merge(
    away_features,
    on = ["match_id", "away_team"],
    how = "left"
)

model_df.shape

# We started with 24 columns, and added 18 home-side features and 18 away-side features.

In [ ]:
# Verify using one match.

model_df[
    [
        "date",
        "home_team",
        "away_team",
        "home_points_last5",
        "away_points_last5",
        "home_avg_xg_for_last5",
        "away_avg_xg_for_last5",
        "home_days_rest",
        "away_days_rest"
    ]
].head(10)

In [ ]:
# Sort chronologically

model_df = model_df.sort_values(
    "date"
).reset_index(drop = True)

model_df[
    [
        "date",
        "home_team",
        "away_team",
        "home_points_last5",
        "away_points_last5"
    ]
].head(15)

In [ ]:
# Check the missing rolling features.

model_df[
    [
        "home_points_last5",
        "away_points_last5",
        "home_avg_xg_for_last5",
        "away_avg_xg_for_last5",
        "home_points_last3_same_venue",
        "away_points_last3_same_venue",
        "home_days_rest",
        "away_days_rest"
    ]
].isnull().sum()

In [ ]:
# Create a column including recent win rate

team_matches["win"] = (
    team_matches["result"] == "W"
).astype("Int64")

In [ ]:
team_matches["win_rate_last5"] = (
    team_matches.groupby("team")["win"]
    .transform(
        lambda x: x.shift(1).rolling(5).mean()
    )
)

In [ ]:
# Inspecting Arsenals' win rate

team_matches[
    team_matches["team"] == "Arsenal"
][
    [
        "date",
        "opponent",
        "result",
        "points_last5",
        "win_rate_last5"
    ]
].head(12)

In [ ]:
model_df[
    [
        "home_team",
        "away_team",
        "home_win_rate_last5",
        "away_win_rate_last5"
    ]
].head(10)

In [ ]:

model_df[
    [
        "date",
        "home_team",
        "away_team",
        "home_win_rate_last5",
        "away_win_rate_last5"
    ]
].dropna().head(10)

In [ ]:
# Look at points over previous 10 matches.
team_matches["points_last10"] = (
    team_matches
    .groupby("team")["points"]
    .transform(
        lambda x: x.shift(1).rolling(10).sum()
    )
)

In [ ]:
# Testing

team_matches[
    team_matches["team"] == "Arsenal"
][
    [
        "date",
        "opponent",
        "result",
        "points_last5",
        "points_last10"
    ]
].head(15)

In [ ]:
model_df[
    [
        "home_team",
        "away_team",
        "home_points_last10",
        "away_points_last10"
    ]
].dropna().head(10)

In [ ]:
# Create a feature for xG difference

team_matches["xg_difference"] = (
    team_matches["xg_for"] - team_matches["xg_against"]
)

In [ ]:
# Look at previous 5 matches

team_matches["xg_difference_last5"] = (
    team_matches
    .groupby("team")["xg_difference"]
    .transform(
        lambda x: x.shift(1).rolling(5).sum()
    )
)

In [ ]:
# Verify using Arsenal

team_matches[
    team_matches["team"] == "Arsenal"
][
    [
        "date",
        "opponent",
        "xg_for",
        "xg_against",
        "xg_difference",
        "xg_difference_last5"
    ]
].head(12)

In [ ]:
model_df[
    [
        "home_team",
        "away_team",
        "home_xg_difference_last5",
        "away_xg_difference_last5"
    ]
].dropna().head(10)

In [ ]:
# Look at how many matches currently have a complete set of the engineered features.

engineered_features = [
    col for col in model_df.columns
    if (
        col.startswith("home_avg_")
        or col.startswith("away_avg_")
        or col.startswith("home_points_")
        or col.startswith("away_points_")
        or col.startswith("home_win_rate_")
        or col.startswith("away_win_rate_")
        or col.startswith("home_goal_difference_")
        or col.startswith("away_goal_difference_")
        or col.startswith("home_xg_difference_")
        or col.startswith("away_xg_difference_")
        or col in ["home_days_rest", "away_days_rest"]
    )
]

print("Number of engineered features:", len(engineered_features))

print(
    "Matches with complete features:",
    model_df[engineered_features].dropna().shape[0]
)

print(
    "Matches with at least one missing feature:",
    model_df[engineered_features].isnull().any(axis = 1).sum()
)

In [ ]:
# Check to see where these features are missing

model_df["has_missing_features"] = (
    model_df[engineered_features]
    .isnull()
    .any(axis = 1)
)

model_df.groupby("season")["has_missing_features"].agg(
    total_matches = "count",
    matches_with_missing = "sum"
)

In [ ]:
# Check to see why the most recent seasons have missing data

model_df[
    (model_df["has_missing_features"]) &
    (model_df["season"] == "2025_26")
][
    [
        "date",
        "matchday",
        "home_team",
        "away_team"
    ]
].sort_values("date")

In [ ]:
team_matches["days_rest"].describe()

In [ ]:
# Getting rid of summer breaks
# They increase the rest days variable drastically

team_matches[
    team_matches["days_rest"] > 30
][
    [
        "date",
        "season",
        "team",
        "opponent",
        "days_rest"
    ]
].head(20)

In [ ]:
team_matches["days_rest"] = (
    team_matches["days_rest"]
    .clip(upper = 14)
)

In [ ]:
team_matches["days_rest"].describe()

In [ ]:
team_matches[
    team_matches["days_rest"] == 14
][
    [
        "date",
        "season",
        "team",
        "opponent",
        "days_rest"
    ]
].head(15)

In [ ]:
# Define the target and build a modeling table using our pre-match features

model_df["target"] = model_df["winner"].map({
    "home": 0,
    "draw": 1,
    "away": 2
})

In [ ]:
model_df[["winner", "target"]].value_counts()

In [ ]:
engineered_features = [
    col for col in model_df.columns
    if (
        col.startswith("home_avg_")
        or col.startswith("away_avg_")
        or col.startswith("home_points_")
        or col.startswith("away_points_")
        or col.startswith("home_win_rate_")
        or col.startswith("away_win_rate_")
        or col.startswith("home_goal_difference_")
        or col.startswith("away_goal_difference_")
        or col.startswith("home_xg_difference_")
        or col.startswith("away_xg_difference_")
        or col in ["home_days_rest", "away_days_rest"]
    )
]

print("Number of model features:", len(engineered_features))
print(engineered_features)

In [ ]:
# Creating a clean dataset
# Keeping matches where all 42 pre-match features and the target result are available.
model_ready_df = model_df.dropna(
    subset = engineered_features + ["target"]
).copy()

print("Original matches:", len(model_df))
print("Model-ready matches:", len(model_ready_df))
print(
    "Removed due to missing features:",
    len(model_df) - len(model_ready_df)
)

In [ ]:
# Grouping matches by season, we don't want to train on 25/26 and test it on 24/25 that would be going backwards.
model_ready_df.groupby("season").size()

In [ ]:
# Conduct a 63/37 train/test split.
# Use seasons 23/24 and 24/25 to predict season 25/26

train_df = model_ready_df[
    model_ready_df["season"].isin(["2023_24", "2024_25"])
].copy()

test_df = model_ready_df[
    model_ready_df["season"] == "2025_26"
].copy()

print("Training matches:", len(train_df))
print("Testing matches:", len(test_df))

In [ ]:
# Save the finished feature-engineered dataset

FEATURE_FILE = PROCESSED_DIR / "model_features.csv"

model_df.to_csv(
    FEATURE_FILE,
    index = False
)

print("Saved:", FEATURE_FILE)
print("Shape:", model_df.shape)

# FEATURE ENGINEERING PART 2

In [ ]:
import pandas as pd
from pathlib import Path

PROCESSED_DIR = Path("../data/processed")

model_df = pd.read_csv(
    PROCESSED_DIR / "model_features.csv"
)

print(model_df.shape)

In [ ]:
# Adding matchup-difference features
# The models performed terribly in terms of predicting draw
# Having a differential between 2 teams might be a better relationship to look at

model_df["xg_form_difference"] = (
    model_df["home_avg_xg_for_last5"]
    - model_df["away_avg_xg_for_last5"]
)

In [ ]:
# Inspect the DataFrame

model_df[
    [
        "home_team",
        "away_team",
        "home_avg_xg_for_last5",
        "away_avg_xg_for_last5",
        "xg_form_difference"
    ]
].dropna().head(10)

In [ ]:
# Adding the focused matchup-difference features together.

model_df["points_form_difference"] = (
    model_df["home_points_last5"]
    - model_df["away_points_last5"]
)

model_df["goal_difference_form_difference"] = (
    model_df["home_goal_difference_last5"]
    - model_df["away_goal_difference_last5"]
)

model_df["xg_difference_form_difference"] = (
    model_df["home_xg_difference_last5"]
    - model_df["away_xg_difference_last5"]
)

model_df["shots_on_target_form_difference"] = (
    model_df["home_avg_shots_on_target_for_last5"]
    - model_df["away_avg_shots_on_target_for_last5"]
)

model_df["win_rate_difference"] = (
    model_df["home_win_rate_last5"]
    - model_df["away_win_rate_last5"]
)

In [ ]:
# Inspect the DataFrame again
# None of this is new info from the match, they're just presenting our existing pre-match info in a way that directly describes the matchup.

model_df[
    [
        "home_team",
        "away_team",
        "xg_form_difference",
        "points_form_difference",
        "goal_difference_form_difference",
        "xg_difference_form_difference",
        "shots_on_target_form_difference",
        "win_rate_difference"
    ]
].dropna().head(10)

In [ ]:
# Saving the new features into the .csv file

FEATURE_FILE = PROCESSED_DIR / "model_features.csv"

model_df.to_csv(
    FEATURE_FILE,
    index = False
)

print("Saved:", FEATURE_FILE)
print("Shape:", model_df.shape)

# FEATURE ENGINEERING PART 3

In [ ]:
import pandas as pd
from pathlib import Path

PROCESSED_DIR = Path("../data/processed")

historical_df = pd.read_csv(
    PROCESSED_DIR / "historical_matches.csv"
)

historical_df["date"] = pd.to_datetime(
    historical_df["date"],
    utc=True
)

model_df = pd.read_csv(
    PROCESSED_DIR / "model_features.csv"
)

print("Historical:", historical_df.shape)
print("Model V2:", model_df.shape)

In [ ]:
# Giving a team an ELO rating might help the model properly predict matches
# A team having a higher ELO (Arsenal) will have a higher chance of beating a team with a lower ELO rating (Burnley)

# Sort all matches in chronological order
elo_matches = historical_df.sort_values(
    ["date", "match_id"]
).reset_index(drop = True)

print(elo_matches.shape)

elo_matches[
    ["date", "season", "home_team", "away_team", "winner"]
].head()

In [ ]:
# Create the ELO calculation

# Starting point
INITIAL_ELO = 1500

# Controls how quickly ratings change
K_FACTOR = 20

# Gives the home team an ELO boost when calculating the expected result
# AKA home field advantage
HOME_ADVANTAGE = 100

elo_ratings = {}
elo_records = []

for _, match in elo_matches.iterrows():
    home_team = match["home_team"]
    away_team = match["away_team"]

    # Give a new team an initial rating of 1500
    home_elo = elo_ratings.get(home_team, INITIAL_ELO)
    away_elo = elo_ratings.get(away_team, INITIAL_ELO)

    # Store Pre-match ratings
    # Happens before the result updates the ratings, preventing daka leakage
    elo_records.append({
        "match_id": match["match_id"],
        "home_elo": home_elo,
        "away_elo": away_elo,
        "elo_difference": home_elo - away_elo
    })

    # Expected result including home advantage
    expected_home = 1 / (
        1 + 10 ** (
            (away_elo - (home_elo + HOME_ADVANTAGE)) / 400
        )
    )

    expected_away = 1 - expected_home

    # Actual result
    if match["winner"] == "home":
        actual_home = 1.0
        actual_away = 0.0

    elif match["winner"] == "away":
        actual_home = 0.0
        actual_away = 1.0

    else:
        actual_home = 0.5
        actual_away = 0.5
    
    # Update ratings after storing pre-match ELO
    elo_ratings[home_team] = (
        home_elo
        + K_FACTOR * (actual_home - expected_home)
    )

    elo_ratings[away_team] = (
        away_elo
        + K_FACTOR * (actual_away - expected_away)
    )

In [ ]:
# Turn this into a DataFrame

elo_df = pd.DataFrame(elo_records)

print(elo_df.shape)
elo_df.head(10)

In [ ]:
# Make sure the ratings start changing after teams play

# This shows Arsenal's first 10 matches chronologically
# We check that Arsenal's first match ELO rating starts at 1500
# ELO changes in later matches based on previous results.
# The opponent also has a changing ELO
# Values shown for a match are still the ratings before that match result
elo_check = elo_matches[
    ["match_id", "date", "home_team", "away_team", "winner"]
].merge(
    elo_df,
    on = "match_id",
    how = "left"
)

elo_check[
    (elo_check["home_team"] == "Arsenal")
    | (elo_check["away_team"] == "Arsenal")
][
    [
        "date",
        "home_team",
        "away_team",
        "winner",
        "home_elo",
        "away_elo",
        "elo_difference"
    ]
].head(10)

In [ ]:
# Merge ELO into V2

model_df = model_df.merge(
    elo_df,
    on = "match_id",
    how = "left"
)

print(model_df.shape)

print(
    model_df[
        ["home_elo", "away_elo", "elo_difference"]
    ].isna().sum()
)

In [ ]:
# Make sure ELO is carrying across seasons 
season_transition_check = model_df[
    model_df["season"].isin(["2023_24", "2024_25"])
][
    [
        "date",
        "season",
        "home_team",
        "away_team",
        "home_elo",
        "away_elo"
    ]
].sort_values("date")

print("Last 5 matchs of 2023/24:")
display(
    season_transition_check[
        season_transition_check["season"] == "2023_24"
    ].tail()
)

print("\nFirst 5 matches of 2024/25:")
display(
    season_transition_check[
        season_transition_check["season"] == "2024_25"
    ].head()
)

In [ ]:
# Save V3
FEATURE_FILE = PROCESSED_DIR / "model_features.csv"

model_df.to_csv(
    FEATURE_FILE,
    index = False
)

print("Saved:", FEATURE_FILE)
print("Shape:", model_df.shape)
print("File size:", FEATURE_FILE.stat().st_size)

# FEATURE ENGINEERING PART 4

In [ ]:
# ELO worked but it could be "more in depth"
# Offseason regression towards 1500
# Lower starting ELO for promoted teams

def build_elo_features(
    matches,
    initial_elo=1500,
    promoted_elo=1450,
    k_factor=20,
    home_advantage=100,
    offseason_regression=0.20
):
    matches = matches.sort_values(
        ["date", "match_id"]
    ).reset_index(drop=True)

    elo_ratings = {}
    team_last_season = {}
    elo_records = []

    for _, match in matches.iterrows():

        season = match["season"]
        home_team = match["home_team"]
        away_team = match["away_team"]

        # HOME TEAM
        if home_team not in elo_ratings:
            home_elo = promoted_elo
            elo_ratings[home_team] = home_elo

        else:
            home_elo = elo_ratings[home_team]

            # First appearance in a new season:
            # regress rating toward 1500
            if team_last_season.get(home_team) != season:
                home_elo = (
                    home_elo
                    + offseason_regression
                    * (initial_elo - home_elo)
                )

                elo_ratings[home_team] = home_elo

        # AWAY TEAM
        if away_team not in elo_ratings:
            away_elo = promoted_elo
            elo_ratings[away_team] = away_elo

        else:
            away_elo = elo_ratings[away_team]

            if team_last_season.get(away_team) != season:
                away_elo = (
                    away_elo
                    + offseason_regression
                    * (initial_elo - away_elo)
                )

                elo_ratings[away_team] = away_elo

        # Store PRE-MATCH Elo
        elo_records.append({
            "match_id": match["match_id"],
            "home_elo_v4": home_elo,
            "away_elo_v4": away_elo,
            "elo_difference_v4": home_elo - away_elo
        })

        # Expected outcome
        expected_home = 1 / (
            1 + 10 ** (
                (away_elo - (home_elo + home_advantage)) / 400
            )
        )

        expected_away = 1 - expected_home

        # Actual outcome
        if match["winner"] == "home":
            actual_home = 1.0
            actual_away = 0.0

        elif match["winner"] == "away":
            actual_home = 0.0
            actual_away = 1.0

        else:
            actual_home = 0.5
            actual_away = 0.5

        # Update AFTER match
        elo_ratings[home_team] = (
            home_elo
            + k_factor * (actual_home - expected_home)
        )

        elo_ratings[away_team] = (
            away_elo
            + k_factor * (actual_away - expected_away)
        )

        team_last_season[home_team] = season
        team_last_season[away_team] = season

    return pd.DataFrame(elo_records)

# In this version, any team appearing for the first time in the dataset gets 1450.

In [ ]:
elo_v4_df = build_elo_features(
    historical_df,
    promoted_elo = 1450,
    offseason_regression = 0.20
)

print(elo_v4_df.shape)
elo_v4_df.head()

In [ ]:
# Verify the offseason regression and new team 1450 rating are working

elo_v4_check = historical_df[
    [
        "match_id",
        "date",
        "season",
        "home_team",
        "away_team"
    ]
].merge(
    elo_v4_df,
    on = "match_id",
    how = "left"
)

elo_v4_check[
    elo_v4_check["season"] == "2024_25"
][
    [
        "date",
        "home_team",
        "away_team",
        "home_elo_v4",
        "away_elo_v4",
        "elo_difference_v4"
    ]
].sort_values("date").head(10)

In [ ]:
# Don't just assume 20% regression towards 1500 is the best option
# Take a look at several other possible values and train it with historical seasons
# Same with promoted teams ELO

promoted_elo_values = [
    1200,
    1225,
    1250,
    1275,
    1300
]

offseason_regression_values = [
    0.35,
    0.40,
    0.45,
    0.50
]

elo_configurations = {}

for promoted_elo in promoted_elo_values:
    for regression in offseason_regression_values:

        config_name = f"elo_{promoted_elo}_reg_{regression}"

        elo_configurations[config_name] = build_elo_features(
            historical_df,
            promoted_elo = promoted_elo,
            offseason_regression = regression
        )

print("Number of configurations:", len(elo_configurations))
print(list(elo_configurations.keys()))

In [ ]:
engineered_features = [
    col for col in model_df.columns
    if (
        col.startswith("home_avg_")
        or col.startswith("away_avg_")
        or col.startswith("home_points_")
        or col.startswith("away_points_")
        or col.startswith("home_win_rate_")
        or col.startswith("away_win_rate_")
        or col.startswith("home_goal_difference_")
        or col.startswith("away_goal_difference_")
        or col.startswith("home_xg_difference_")
        or col.startswith("away_xg_difference_")
        or col in ["home_days_rest", "away_days_rest"]
    )
]

differential_features = [
    "xg_form_difference",
    "points_form_difference",
    "goal_difference_form_difference",
    "xg_difference_form_difference",
    "shots_on_target_form_difference",
    "win_rate_difference"
]

engineered_features_v2 = engineered_features + differential_features

print("V1 features:", len(engineered_features))
print("V2 features:", len(engineered_features_v2))

In [ ]:
# Score the configurations by using 2023/24 for training and 2024/25 for validation
# Use RF structure and same V2 features since the ELO settings are being tested
# Essentially trainining 20 Random Forest models to see which configuration fits best.

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, log_loss

validation_results = []

for config_name, elo_config_df in elo_configurations.items():

    # Merge this Elo configuration with our V2 dataset
    temp_df = model_df.drop(
        columns=[
            "home_elo",
            "away_elo",
            "elo_difference"
        ],
        errors="ignore"
    ).merge(
        elo_config_df,
        on="match_id",
        how="left"
    )

    # V4 Elo column names
    elo_features_temp = [
        "home_elo_v4",
        "away_elo_v4",
        "elo_difference_v4"
    ]

    features_temp = (
        engineered_features_v2
        + elo_features_temp
    )

    # Only complete rows
    temp_ready = temp_df.dropna(
        subset=features_temp + ["target"]
    ).copy()

    # 2023/24 = training
    train_temp = temp_ready[
        temp_ready["season"] == "2023_24"
    ]

    # 2024/25 = validation
    validation_temp = temp_ready[
        temp_ready["season"] == "2024_25"
    ]

    X_train_temp = train_temp[features_temp]
    y_train_temp = train_temp["target"]

    X_val_temp = validation_temp[features_temp]
    y_val_temp = validation_temp["target"]

    # Same Random Forest settings
    temp_model = RandomForestClassifier(
        n_estimators=500,
        random_state=42,
        n_jobs=-1
    )

    temp_model.fit(
        X_train_temp,
        y_train_temp
    )

    val_pred = temp_model.predict(X_val_temp)
    val_proba = temp_model.predict_proba(X_val_temp)

    validation_results.append({
        "configuration": config_name,
        "accuracy": accuracy_score(
            y_val_temp,
            val_pred
        ),
        "log_loss": log_loss(
            y_val_temp,
            val_proba
        )
    })

In [ ]:
# Create a list showing which configuration had the lower log loss

validation_results_df = pd.DataFrame(
    validation_results
).sort_values(
    "log_loss"
).reset_index(drop = True)

validation_results_df

In [ ]:
# Build the V4 ELO

elo_v4_final = build_elo_features(
    historical_df,
    promoted_elo = 1200,
    offseason_regression = 0.50
)

print(elo_v4_final.shape)
elo_v4_final.head()

In [ ]:
# Merge into model_df

model_df = model_df.merge(
    elo_v4_final,
    on = "match_id",
    how = "left"
)

print(model_df.shape)

print(
    model_df[
        [
            "home_elo",
            "away_elo",
            "elo_difference",
            "home_elo_v4",
            "away_elo_v4",
            "elo_difference_v4"
        ]
    ].isna().sum()
)

In [ ]:
# Save the new feature file

FEATURE_FILE = PROCESSED_DIR / "model_features.csv"

model_df.to_csv(
    FEATURE_FILE,
    index = False
)

print("Saved:", FEATURE_FILE)
print("Shape:", model_df.shape)

# FEATURE ENGINEERING PART 5

In [ ]:
elo_v4b = build_elo_features(
    historical_df,
    promoted_elo=1400,
    offseason_regression=0.25
)

elo_v4b = elo_v4b.rename(columns={
    "home_elo_v4": "home_elo_v4b",
    "away_elo_v4": "away_elo_v4b",
    "elo_difference_v4": "elo_difference_v4b"
})

print(elo_v4b.shape)
elo_v4b.head()

In [ ]:
# Merge V4b into model_df

model_df = model_df.merge(
    elo_v4b,
    on = "match_id",
    how = "left"
)

print(model_df.shape)

print(
    model_df[
        [
            "home_elo_v4b",
            "away_elo_v4b",
            "elo_difference_v4b"
        ]
    ].isna().sum()
)

In [ ]:
# Save

FEATURE_FILE = PROCESSED_DIR / "model_features.csv"

model_df.to_csv(
    FEATURE_FILE,
    index = False
)

print("Saved:", FEATURE_FILE)
print("Shape:", model_df.shape)

# FEATURE ENGINEERING PART 6

In [ ]:
# Do same process, with different configuration

elo_v4c = build_elo_features(
    historical_df,
    promoted_elo=1300,
    offseason_regression=0.20
)

elo_v4c = elo_v4c.rename(columns={
    "home_elo_v4": "home_elo_v4c",
    "away_elo_v4": "away_elo_v4c",
    "elo_difference_v4": "elo_difference_v4c"
})

print(elo_v4c.shape)
elo_v4c.head()

In [ ]:
model_df = model_df.merge(
    elo_v4c,
    on="match_id",
    how="left"
)

print(model_df.shape)

print(
    model_df[
        [
            "home_elo_v4c",
            "away_elo_v4c",
            "elo_difference_v4c"
        ]
    ].isna().sum()
)

In [ ]:
FEATURE_FILE = PROCESSED_DIR / "model_features.csv"

model_df.to_csv(
    FEATURE_FILE,
    index=False
)

print("Saved:", FEATURE_FILE)
print("Shape:", model_df.shape)

# INCORPORATING ALL LEAGUES 

In [ ]:
# Load new historical dataset

import pandas as pd
import numpy as np
from pathlib import Path

PROCESSED_DIR = Path("../data/processed")

historical_df = pd.read_csv(
    PROCESSED_DIR / "historical_matches.csv"
)

historical_df["date"] = pd.to_datetime(
    historical_df["date"],
    utc=True
)

print("Shape:", historical_df.shape)
print()
print(historical_df.dtypes)

In [ ]:
# Make sure matches are in chronological order across all competitions

historical_df = historical_df.sort_values(
    ["date", "match_id"]
).reset_index(drop = True)

print(historical_df[[
    "date",
    "competition",
    "home_team",
    "away_team"
]].head(10))

In [ ]:
# Convert match-level data into team-level rows.
# Each match will eventually become two rows: home team's perspective and away team's perspective

home_rows = historical_df[[
    "match_id",
    "date",
    "season",
    "competition",
    "matchday",
    "home_team",
    "away_team",
    "home_goals",
    "away_goals",
    "home_xg",
    "away_xg",
    "home_shots",
    "away_shots",
    "home_shots_on_target",
    "away_shots_on_target",
    "home_possession",
    "away_possession",
    "home_passes",
    "away_passes",
    "home_corners",
    "away_corners"
]].copy()

home_rows = home_rows.rename(columns={
    "home_team": "team",
    "away_team": "opponent",
    "home_goals": "goals_for",
    "away_goals": "goals_against",
    "home_xg": "xg_for",
    "away_xg": "xg_against",
    "home_shots": "shots_for",
    "away_shots": "shots_against",
    "home_shots_on_target": "shots_on_target_for",
    "away_shots_on_target": "shots_on_target_against",
    "home_possession": "possession_for",
    "away_possession": "possession_against",
    "home_passes": "passes_for",
    "away_passes": "passes_against",
    "home_corners": "corners_for",
    "away_corners": "corners_against"
})

home_rows["venue"] = "home"

print(home_rows.shape)
print()
print(home_rows.head())

In [ ]:
away_rows = historical_df[[
    "match_id",
    "date",
    "season",
    "competition",
    "matchday",
    "away_team",
    "home_team",
    "away_goals",
    "home_goals",
    "away_xg",
    "home_xg",
    "away_shots",
    "home_shots",
    "away_shots_on_target",
    "home_shots_on_target",
    "away_possession",
    "home_possession",
    "away_passes",
    "home_passes",
    "away_corners",
    "home_corners"
]].copy()

away_rows = away_rows.rename(columns={
    "away_team": "team",
    "home_team": "opponent",
    "away_goals": "goals_for",
    "home_goals": "goals_against",
    "away_xg": "xg_for",
    "home_xg": "xg_against",
    "away_shots": "shots_for",
    "home_shots": "shots_against",
    "away_shots_on_target": "shots_on_target_for",
    "home_shots_on_target": "shots_on_target_against",
    "away_possession": "possession_for",
    "home_possession": "possession_against",
    "away_passes": "passes_for",
    "home_passes": "passes_against",
    "away_corners": "corners_for",
    "home_corners": "corners_against"
})

away_rows["venue"] = "away"

print(away_rows.shape)
print()
print(away_rows.head())

In [ ]:
# Combine home and away teams into full team-level dataset
# Gives one row per team per match, best structure to use when dealing with rolling form, rest days, congestion, and ELO

team_matches = pd.concat(
    [home_rows, away_rows],
    ignore_index = True
)

team_matches = team_matches.sort_values(
    ["team", "date", "match_id"]
).reset_index(drop = True)

print("Shape:", team_matches.shape)
print()
print(team_matches.head(10))

In [ ]:
# Add simplest pre-match features first
# Result, points, GD, rest days

team_matches["result"] = np.select(
    [
        team_matches["goals_for"] > team_matches["goals_against"],
        team_matches["goals_for"] == team_matches["goals_against"]
    ],
    [
        "W",
        "D"
    ],
    default="L"
)

team_matches["points"] = np.select(
    [
        team_matches["result"] == "W",
        team_matches["result"] == "D"
    ],
    [
        3,
        1
    ],
    default=0
)

team_matches["goal_difference"] = (
    team_matches["goals_for"] - team_matches["goals_against"]
)

team_matches["days_rest"] = (
    team_matches.groupby("team")["date"]
    .diff()
    .dt.total_seconds()
    .div(86400)
)

print(team_matches[[
    "team",
    "date",
    "competition",
    "result",
    "points",
    "goal_difference",
    "days_rest"
]].head(12))

In [ ]:
# Create first all-competition rolling features
# Preventing data leakage

rolling_cols = [
    "points",
    "goals_for",
    "goals_against",
    "goal_difference"
]

for col in rolling_cols:
    team_matches[f"{col}_last5_all"] = (
        team_matches.groupby("team")[col]
        .transform(lambda x: x.shift(1).rolling(5, min_periods=1).mean())
    )

print(team_matches[[
    "team",
    "date",
    "competition",
    "points",
    "points_last5_all",
    "goal_difference_last5_all"
]].head(10))

In [ ]:
# Add all competition win rate over previous 5 matches

team_matches["win"] = (
    team_matches["result"] == "W"
).astype(int)

team_matches["win_rate_last5_all"] = (
    team_matches.groupby("team")["win"]
    .transform(lambda x: x.shift(1).rolling(5, min_periods=1).mean())
)

print(team_matches[[
    "team",
    "date",
    "result",
    "win",
    "points_last5_all",
    "win_rate_last5_all"
]].head(10))

In [ ]:
# Add performance-stat rolling features for previous 5 matches.

performance_cols = [
    "xg_for",
    "xg_against",
    "shots_for",
    "shots_against",
    "shots_on_target_for",
    "shots_on_target_against",
    "possession_for",
    "passes_for",
    "corners_for"
]

for col in performance_cols:
    team_matches[f"{col}_last5_all"] = (
        team_matches.groupby("team")[col]
        .transform(
            lambda x: x.shift(1).rolling(
                5,
                min_periods=1
            ).mean()
        )
    )

print(team_matches[[
    "team",
    "date",
    "xg_for",
    "xg_for_last5_all",
    "shots_for_last5_all",
    "shots_on_target_for_last5_all"
]].head(10))

In [ ]:
# Create two more rolling features: xG difference and shots-on-target difference

team_matches["xg_difference_last5_all"] = (
    team_matches["xg_for_last5_all"]
    - team_matches["xg_against_last5_all"]
)

team_matches["shots_on_target_difference_last5_all"] = (
    team_matches["shots_on_target_for_last5_all"]
    - team_matches["shots_on_target_against_last5_all"]
)

print(team_matches[[
    "team",
    "date",
    "xg_for_last5_all",
    "xg_against_last5_all",
    "xg_difference_last5_all",
    "shots_on_target_difference_last5_all"
]].head(10))

In [ ]:
# Separate Premier League-only form an all-competition form

pl_team_matches = team_matches[
    team_matches["competition"] == "premier_league"
].copy()

pl_team_matches = pl_team_matches.sort_values(
    ["team", "date", "match_id"]
).reset_index(drop=True)

print("PL team rows:", pl_team_matches.shape)

print(pl_team_matches[[
    "team",
    "date",
    "season",
    "competition",
    "points"
]].head(10))

In [ ]:
pl_rolling_cols = [
    "points",
    "goals_for",
    "goals_against",
    "goal_difference"
]

for col in pl_rolling_cols:
    pl_team_matches[f"{col}_last5_pl"] = (
        pl_team_matches.groupby("team")[col]
        .transform(
            lambda x: x.shift(1).rolling(
                5,
                min_periods=1
            ).mean()
        )
    )

print(pl_team_matches[[
    "team",
    "date",
    "points",
    "points_last5_pl",
    "goal_difference_last5_pl"
]].head(10))

In [ ]:
# Add PL-only performance stats

pl_performance_cols = [
    "xg_for",
    "xg_against",
    "shots_for",
    "shots_against",
    "shots_on_target_for",
    "shots_on_target_against",
    "possession_for",
    "passes_for",
    "corners_for"
]

for col in pl_performance_cols:
    pl_team_matches[f"{col}_last5_pl"] = (
        pl_team_matches.groupby("team")[col]
        .transform(
            lambda x: x.shift(1).rolling(
                5,
                min_periods=1
            ).mean()
        )
    )

print(pl_team_matches[[
    "team",
    "date",
    "xg_for",
    "xg_for_last5_pl",
    "shots_for_last5_pl",
    "shots_on_target_for_last5_pl"
]].head(10))

In [ ]:
# Add PL-only win rate and difference features

pl_team_matches["win_rate_last5_pl"] = (
    pl_team_matches.groupby("team")["win"]
    .transform(
        lambda x: x.shift(1).rolling(
            5,
            min_periods=1
        ).mean()
    )
)

pl_team_matches["xg_difference_last5_pl"] = (
    pl_team_matches["xg_for_last5_pl"]
    - pl_team_matches["xg_against_last5_pl"]
)

pl_team_matches["shots_on_target_difference_last5_pl"] = (
    pl_team_matches["shots_on_target_for_last5_pl"]
    - pl_team_matches["shots_on_target_against_last5_pl"]
)

print(pl_team_matches[[
    "team",
    "date",
    "points_last5_pl",
    "win_rate_last5_pl",
    "xg_difference_last5_pl",
    "shots_on_target_difference_last5_pl"
]].head(10))

In [ ]:
check_cols = [
    "team",
    "date",
    "competition",
    "points_last5_all",
    "points_last5_pl",
    "win_rate_last5_all",
    "win_rate_last5_pl",
    "goal_difference_last5_all",
    "goal_difference_last5_pl"
]

print(pl_team_matches[check_cols].head(10))

In [ ]:
# Add fixture congesion features from all-competition schedule

def count_recent_matches(group, days):
    dates = group["date"]

    return dates.apply(
        lambda current_date: (
            (dates < current_date)
            & (dates >= current_date - pd.Timedelta(days=days))
        ).sum()
    )

team_matches["matches_last_7_days"] = (
    team_matches.groupby("team", group_keys=False)
    .apply(lambda group: count_recent_matches(group, 7))
)

team_matches["matches_last_14_days"] = (
    team_matches.groupby("team", group_keys=False)
    .apply(lambda group: count_recent_matches(group, 14))
)

print(team_matches[[
    "team",
    "date",
    "competition",
    "days_rest",
    "matches_last_7_days",
    "matches_last_14_days"
]].head(12))

In [ ]:
# Add congestion features on the PL rows

congestion_features = team_matches[[
    "match_id",
    "team",
    "matches_last_7_days",
    "matches_last_14_days"
]].copy()

pl_team_matches = pl_team_matches.merge(
    congestion_features,
    on=["match_id", "team"],
    how="left"
)

print(pl_team_matches[[
    "team",
    "date",
    "competition",
    "days_rest",
    "matches_last_7_days",
    "matches_last_14_days"
]].head(10))

In [ ]:
# Check Arsenal's schedule

arsenal_schedule = team_matches[
    (team_matches["team"] == "Arsenal") &
    (team_matches["date"] >= "2020-09-20") &
    (team_matches["date"] <= "2020-10-05")
][[
    "date",
    "competition",
    "opponent",
    "venue",
    "result"
]]

print(arsenal_schedule)

In [ ]:
# Add a 10-match PL form window

pl_last10_cols = [
    "points",
    "goal_difference"
]

for col in pl_last10_cols:
    pl_team_matches[f"{col}_last10_pl"] = (
        pl_team_matches.groupby("team")[col]
        .transform(
            lambda x: x.shift(1).rolling(
                10,
                min_periods=1
            ).mean()
        )
    )

print(pl_team_matches[[
    "team",
    "date",
    "points_last5_pl",
    "points_last10_pl",
    "goal_difference_last5_pl",
    "goal_difference_last10_pl"
]].head(12))

In [ ]:
# Add 10 match window for win rate

pl_team_matches["win_rate_last10_pl"] = (
    pl_team_matches.groupby("team")["win"]
    .transform(
        lambda x: x.shift(1).rolling(
            10,
            min_periods=1
        ).mean()
    )
)

print(pl_team_matches[[
    "team",
    "date",
    "win_rate_last5_pl",
    "win_rate_last10_pl"
]].head(12))

In [ ]:
# Work towards ELO/opponent-strength side

elo_matches = historical_df[[
    "match_id",
    "date",
    "season",
    "competition",
    "home_team",
    "away_team",
    "home_goals",
    "away_goals"
]].copy()

elo_matches = elo_matches.sort_values(
    ["date", "match_id"]
).reset_index(drop=True)

print("Elo matches:", elo_matches.shape)
print()
print(elo_matches.head(10))

In [ ]:
# Initialize ELO system
# Start every team at 1500 rating 

INITIAL_ELO = 1500
K_FACTOR = 20

elo_ratings = {}

all_teams = pd.concat([
    elo_matches["home_team"],
    elo_matches["away_team"]
]).unique()

for team in all_teams:
    elo_ratings[team] = INITIAL_ELO

print("Number of teams:", len(elo_ratings))
print()

for team in list(elo_ratings)[:10]:
    print(team, elo_ratings[team])

In [ ]:
# Create expected-result function, accounts for opponent strength

def expected_score(team_elo, opponent_elo):
    return 1 / (
        1 + 10 ** ((opponent_elo - team_elo) / 400)
    )

print("Equal teams:", expected_score(1500, 1500))
print("Stronger team:", expected_score(1700, 1500))
print("Weaker team:", expected_score(1500, 1700))

In [ ]:
# Add actual result score for ELO updates

def actual_score(home_goals, away_goals):
    if home_goals > away_goals:
        return 1.0, 0.0
    elif home_goals < away_goals:
        return 0.0, 1.0
    else:
        return 0.5, 0.5

print("Home win:", actual_score(2, 1))
print("Draw:", actual_score(1, 1))
print("Away win:", actual_score(0, 2))

In [ ]:
# Test one actual ELO update

test_match = elo_matches.iloc[0]

home_team = test_match["home_team"]
away_team = test_match["away_team"]

home_elo = elo_ratings[home_team]
away_elo = elo_ratings[away_team]

home_expected = expected_score(home_elo, away_elo)
away_expected = expected_score(away_elo, home_elo)

home_actual, away_actual = actual_score(
    test_match["home_goals"],
    test_match["away_goals"]
)

new_home_elo = home_elo + K_FACTOR * (
    home_actual - home_expected
)

new_away_elo = away_elo + K_FACTOR * (
    away_actual - away_expected
)

print(home_team, home_elo, "->", new_home_elo)
print(away_team, away_elo, "->", new_away_elo)

In [ ]:
# Run ELO chronologically across all matches
# Need to save each team's ELO before the match happens (preventing leakage)

elo_ratings = {}

elo_history = []

for _, match in elo_matches.iterrows():
    home_team = match["home_team"]
    away_team = match["away_team"]

    home_elo = elo_ratings.get(home_team, INITIAL_ELO)
    away_elo = elo_ratings.get(away_team, INITIAL_ELO)

    home_expected = expected_score(home_elo, away_elo)
    away_expected = expected_score(away_elo, home_elo)

    home_actual, away_actual = actual_score(
        match["home_goals"],
        match["away_goals"]
    )

    # Save PRE-MATCH Elo ratings
    elo_history.append({
        "match_id": match["match_id"],
        "home_elo": home_elo,
        "away_elo": away_elo,
        "elo_difference": home_elo - away_elo
    })

    # Update ratings AFTER the match
    elo_ratings[home_team] = (
        home_elo
        + K_FACTOR * (home_actual - home_expected)
    )

    elo_ratings[away_team] = (
        away_elo
        + K_FACTOR * (away_actual - away_expected)
    )

elo_history_df = pd.DataFrame(elo_history)

print("Elo history shape:", elo_history_df.shape)
print()
print(elo_history_df.head(10))

In [ ]:
# Attach the pre-match ELO ratings to the Premier League dataset

pl_team_matches = pl_team_matches.merge(
    elo_history_df,
    on = "match_id",
    how = "left"
)

print(pl_team_matches[[
    "team",
    "date",
    "venue",
    "home_elo",
    "away_elo",
    "elo_difference"
]].head(10))

In [ ]:
# Convert ELO into team's perspective

pl_team_matches["team_elo"] = np.where(
    pl_team_matches["venue"] == "home",
    pl_team_matches["home_elo"],
    pl_team_matches["away_elo"]
)

pl_team_matches["opponent_elo"] = np.where(
    pl_team_matches["venue"] == "home",
    pl_team_matches["away_elo"],
    pl_team_matches["home_elo"]
)

pl_team_matches["team_elo_difference"] = (
    pl_team_matches["team_elo"]
    - pl_team_matches["opponent_elo"]
)

print(pl_team_matches[[
    "team",
    "date",
    "venue",
    "team_elo",
    "opponent_elo",
    "team_elo_difference"
]].head(10))

In [ ]:
# Inspect the highest-rated teams at the end of our historical period
# Helps make sure the ELO system is producing correct rankings

final_elo_df = pd.DataFrame(
    elo_ratings.items(),
    columns=["team", "elo"]
).sort_values(
    "elo",
    ascending=False
).reset_index(drop=True)

print(final_elo_df.head(20))

In [ ]:
# Check coverage, some PL clubs are above clubs like Madrid or Barcelona

team_match_counts = (
    team_matches.groupby("team")
    .size()
    .reset_index(name="matches_in_dataset")
)

final_elo_check = final_elo_df.merge(
    team_match_counts,
    on="team",
    how="left"
)

print(final_elo_check.head(20))

In [ ]:
# Check how much coverage the PL clubs have

pl_teams = pd.unique(
    pd.concat([
        historical_df.loc[
            historical_df["competition"] == "premier_league",
            "home_team"
        ],
        historical_df.loc[
            historical_df["competition"] == "premier_league",
            "away_team"
        ]
    ])
)

pl_elo_check = final_elo_check[
    final_elo_check["team"].isin(pl_teams)
].sort_values(
    "matches_in_dataset",
    ascending=False
)

print("Unique PL teams:", len(pl_elo_check))
print()
print(pl_elo_check.to_string(index=False))

In [ ]:
# Check if a promoted team's ELO rating carry from the Championship into the first PL match

brentford_transition = team_matches[
    (team_matches["team"] == "Brentford") &
    (team_matches["date"] >= "2021-05-01") &
    (team_matches["date"] <= "2021-08-20")
][[
    "date",
    "season",
    "competition",
    "opponent",
    "result"
]]

print(brentford_transition.to_string(index=False))

In [ ]:
brentford_first_pl = pl_team_matches[
    (pl_team_matches["team"] == "Brentford") &
    (pl_team_matches["season"] == "2021_22")
][[
    "date",
    "opponent",
    "venue",
    "team_elo",
    "opponent_elo",
    "team_elo_difference"
]].head(3)

print(brentford_first_pl)

In [ ]:
# Check another promoted team

forest_first_pl = pl_team_matches[
    (pl_team_matches["team"] == "Nottingham Forest") &
    (pl_team_matches["season"] == "2022_23")
][[
    "date",
    "opponent",
    "venue",
    "team_elo",
    "opponent_elo",
    "team_elo_difference"
]].head(3)

print(forest_first_pl)

In [ ]:
# Build the match-level modeling dataset
# Model needs one row per match, with the home team's pre-match features and away team's pre match features side by side


# Separate those rows
pl_home_features = pl_team_matches[
    pl_team_matches["venue"] == "home"
].copy()

pl_away_features = pl_team_matches[
    pl_team_matches["venue"] == "away"
].copy()

print("Home rows:", pl_home_features.shape)
print("Away rows:", pl_away_features.shape)

print()
print(pl_home_features[[
    "match_id",
    "date",
    "team",
    "opponent"
]].head())

print()
print(pl_away_features[[
    "match_id",
    "date",
    "team",
    "opponent"
]].head())

In [ ]:
# Verify that each match_id occurs only once in each DataFrame before merging

home_match_ids = set(pl_home_features["match_id"])
away_match_ids = set(pl_away_features["match_id"])

print("Unique home match IDs:", len(home_match_ids))
print("Unique away match IDs:", len(away_match_ids))
print("Match IDs in both:", len(home_match_ids & away_match_ids))

print("Only in home:", len(home_match_ids - away_match_ids))
print("Only in away:", len(away_match_ids - home_match_ids))

In [ ]:
# Build the one row per match dataset
# Merge two sides by match_id

pl_match_features = pl_home_features.merge(
    pl_away_features,
    on = "match_id",
    how = "inner",
    suffixes = ("_home", "_away")
)

print("Match-level shape:", pl_match_features.shape)

print(
    pl_match_features[[
        "match_id",
        "date_home",
        "team_home",
        "team_away"
    ]].head()
)

In [ ]:
# Before selecting model features, create the target variable: what happened in each match
# Each row has both sides' current-match info, so use the home-side goals only to create the target and not as an input feature

pl_match_features["target"] = np.select(
    [
        pl_match_features["goals_for_home"] > pl_match_features["goals_against_home"],
        pl_match_features["goals_for_home"] == pl_match_features["goals_against_home"]
    ],
    ["H", "D"],
    default="A"
)

print(
    pl_match_features[[
        "date_home",
        "team_home",
        "team_away",
        "goals_for_home",
        "goals_against_home",
        "target"
    ]].head(10)
)

print()
print(pl_match_features["target"].value_counts())
print()
print(pl_match_features["target"].value_counts(normalize=True))

In [ ]:
# Verify seasons before feature selection
# Make sure all seasons have 380 matches

season_counts = (
    pl_match_features["season_home"]
    .value_counts()
    .sort_index()
)

print(season_counts)
print()
print("Total:", season_counts.sum())

In [ ]:
# Decide which of the 139 columns are reliable pre-match predictors

for i, col in enumerate(pl_match_features.columns):
    print(i, col)

In [ ]:
# Avoid current match stats such as result_home, goals_for_home in order to prevent data leakage

# Create a clean feature list
model_features = [
    # Home team - all competitions form
    "points_last5_all_home",
    "goal_difference_last5_all_home",
    "win_rate_last5_all_home",
    "xg_difference_last5_all_home",
    "shots_on_target_difference_last5_all_home",

    # Away team - all competitions form
    "points_last5_all_away",
    "goal_difference_last5_all_away",
    "win_rate_last5_all_away",
    "xg_difference_last5_all_away",
    "shots_on_target_difference_last5_all_away",

    # Home team - Premier League form
    "points_last5_pl_home",
    "goal_difference_last5_pl_home",
    "win_rate_last5_pl_home",
    "xg_difference_last5_pl_home",
    "shots_on_target_difference_last5_pl_home",
    "points_last10_pl_home",
    "goal_difference_last10_pl_home",
    "win_rate_last10_pl_home",

    # Away team - Premier League form
    "points_last5_pl_away",
    "goal_difference_last5_pl_away",
    "win_rate_last5_pl_away",
    "xg_difference_last5_pl_away",
    "shots_on_target_difference_last5_pl_away",
    "points_last10_pl_away",
    "goal_difference_last10_pl_away",
    "win_rate_last10_pl_away",

    # Rest and fixture congestion
    "days_rest_home",
    "days_rest_away",
    "matches_last_7_days_home",
    "matches_last_7_days_away",
    "matches_last_14_days_home",
    "matches_last_14_days_away",

    # Pre-match Elo
    "team_elo_home",
    "team_elo_away",
    "team_elo_difference_home"
]

print("Number of model features:", len(model_features))

print()
print(pl_match_features[model_features].dtypes)

In [ ]:
# Check how much data is missing in each feature

missing_features = (
    pl_match_features[model_features]
    .isna()
    .sum()
    .sort_values(ascending=False)
)

missing_percent = (
    pl_match_features[model_features]
    .isna()
    .mean()
    .mul(100)
    .sort_values(ascending=False)
)

missing_summary = pd.DataFrame({
    "missing_count": missing_features,
    "missing_percent": missing_percent
})

print(missing_summary)

In [ ]:
# For the first baseline model, leave out xG since it's missing 33%

baseline_features = [
    feature
    for feature in model_features
    if "xg_" not in feature
]

print("Baseline features:", len(baseline_features))

print()
for feature in baseline_features:
    print(feature)

In [ ]:
# Check which matches are missing some of the baseline features

missing_baseline_rows = pl_match_features[
    pl_match_features[baseline_features]
    .isna()
    .any(axis=1)
]

print("Matches with at least one missing baseline feature:",
      len(missing_baseline_rows))

print()

print(
    missing_baseline_rows[[
        "date_home",
        "season_home",
        "team_home",
        "team_away"
    ]]
    .sort_values("date_home")
    .to_string(index=False)
)

In [ ]:
# Find out what's missing in those 20 matches

for _, row in missing_baseline_rows.sort_values("date_home").iterrows():
    missing = [
        feature
        for feature in baseline_features
        if pd.isna(row[feature])
    ]

    print(
        row["date_home"].date(),
        "|",
        row["team_home"],
        "vs",
        row["team_away"]
    )

    print("Missing:", missing)
    print()

In [ ]:
# For the baseline, cleanest approach is to use median imputation fitted on the training data later

# Create CSV file with processed match dataset
FEATURES_PATH = PROCESSED_DIR / "pl_match_features.csv"

pl_match_features.to_csv(
    FEATURES_PATH,
    index=False
)

print("Saved:", FEATURES_PATH)
print("Shape:", pl_match_features.shape)

In [ ]:
historical_df = pd.read_csv(
    "../data/processed/historical_matches.csv"
)

historical_df["date"] = pd.to_datetime(
    historical_df["date"],
    utc=True
)

pl_matches = historical_df[
    historical_df["competition"]
    == "premier_league"
].copy()


# Merge original pre-match Elo ratings
pl_matches = pl_matches.merge(
    elo_history_df,
    on="match_id",
    how="left"
)


home_team_rows = pd.DataFrame({
    "match_id": pl_matches["match_id"],
    "date": pl_matches["date"],
    "season": pl_matches["season"],

    "team": pl_matches["home_team"],
    "opponent": pl_matches["away_team"],

    "venue": "home",

    "goals_for": pl_matches["home_goals"],
    "goals_against": pl_matches["away_goals"],

    "team_elo": pl_matches["home_elo"],
    "opponent_elo": pl_matches["away_elo"]
})


away_team_rows = pd.DataFrame({
    "match_id": pl_matches["match_id"],
    "date": pl_matches["date"],
    "season": pl_matches["season"],

    "team": pl_matches["away_team"],
    "opponent": pl_matches["home_team"],

    "venue": "away",

    "goals_for": pl_matches["away_goals"],
    "goals_against": pl_matches["home_goals"],

    "team_elo": pl_matches["away_elo"],
    "opponent_elo": pl_matches["home_elo"]
})


pl_team_matches = pd.concat(
    [
        home_team_rows,
        away_team_rows
    ],
    ignore_index=True
)


pl_team_matches = (
    pl_team_matches
    .sort_values(
        [
            "team",
            "date",
            "match_id"
        ]
    )
    .reset_index(drop=True)
)


print(
    pl_team_matches.shape
)

print()

print(
    pl_team_matches.head()
)

print()

print(
    "Missing opponent Elo:",
    pl_team_matches[
        "opponent_elo"
    ].isna().sum()
)

In [ ]:
pl_team_matches[
    "opponent_elo_last5_pl"
] = (
    pl_team_matches
    .groupby("team")[
        "opponent_elo"
    ]
    .transform(
        lambda x: (
            x.shift(1)
            .rolling(
                window=5,
                min_periods=1
            )
            .mean()
        )
    )
)


print(
    pl_team_matches[
        [
            "team",
            "date",
            "opponent",
            "opponent_elo",
            "opponent_elo_last5_pl"
        ]
    ].head(12)
)

print()

print(
    "Missing values:",
    pl_team_matches[
        "opponent_elo_last5_pl"
    ].isna().sum()
)

In [ ]:
home_opponent_strength = (
    pl_team_matches[
        pl_team_matches["venue"] == "home"
    ][
        [
            "match_id",
            "opponent_elo_last5_pl"
        ]
    ]
    .rename(
        columns={
            "opponent_elo_last5_pl":
            "opponent_elo_last5_pl_home"
        }
    )
)


away_opponent_strength = (
    pl_team_matches[
        pl_team_matches["venue"] == "away"
    ][
        [
            "match_id",
            "opponent_elo_last5_pl"
        ]
    ]
    .rename(
        columns={
            "opponent_elo_last5_pl":
            "opponent_elo_last5_pl_away"
        }
    )
)


opponent_strength_features = (
    home_opponent_strength
    .merge(
        away_opponent_strength,
        on="match_id",
        how="inner"
    )
)


print(
    opponent_strength_features.shape
)

print()

print(
    opponent_strength_features.head(10)
)

print()

print(
    "Duplicate match IDs:",
    opponent_strength_features[
        "match_id"
    ].duplicated().sum()
)

print(
    "Missing home:",
    opponent_strength_features[
        "opponent_elo_last5_pl_home"
    ].isna().sum()
)

print(
    "Missing away:",
    opponent_strength_features[
        "opponent_elo_last5_pl_away"
    ].isna().sum()
)

In [ ]:
opponent_strength_features.to_csv(
    "../data/processed/opponent_strength_features.csv",
    index=False
)

print(
    "Saved:",
    opponent_strength_features.shape
)

In [ ]:
# Build match expectation features
# Giving a form relative to expected performance view

pl_team_matches[
    "elo_expected_score"
] = (
    1
    / (
        1
        + 10 ** (
            (
                pl_team_matches["opponent_elo"]
                - pl_team_matches["team_elo"]
            )
            / 400
        )
    )
)


pl_team_matches[
    "actual_score"
] = np.select(
    [
        pl_team_matches["goals_for"]
        > pl_team_matches["goals_against"],

        pl_team_matches["goals_for"]
        == pl_team_matches["goals_against"]
    ],
    [
        1.0,
        0.5
    ],
    default=0.0
)


pl_team_matches[
    "elo_performance_residual"
] = (
    pl_team_matches["actual_score"]
    - pl_team_matches["elo_expected_score"]
)


print(
    pl_team_matches[
        [
            "team",
            "opponent",
            "team_elo",
            "opponent_elo",
            "elo_expected_score",
            "actual_score",
            "elo_performance_residual"
        ]
    ].head(15)
)

print()

print(
    pl_team_matches[
        "elo_performance_residual"
    ].describe()
)

In [ ]:
# Turn this into recent form vs expectation
# One surprising result doesn't mean a team is genuinely improving
# Averaging the previous 5 residuals gives a smoother measure of whether a team has consistently performed above or below what the underlying ELO strength predicted

pl_team_matches[
    "elo_performance_last5_pl"
] = (
    pl_team_matches
    .groupby("team")[
        "elo_performance_residual"
    ]
    .transform(
        lambda x: (
            x.shift(1)
            .rolling(
                window=5,
                min_periods=1
            )
            .mean()
        )
    )
)


print(
    pl_team_matches[
        [
            "team",
            "date",
            "opponent",
            "elo_expected_score",
            "actual_score",
            "elo_performance_residual",
            "elo_performance_last5_pl"
        ]
    ].head(15)
)

print()

print(
    "Missing values:",
    pl_team_matches[
        "elo_performance_last5_pl"
    ].isna().sum()
)

print()

print(
    pl_team_matches[
        "elo_performance_last5_pl"
    ].describe()
)

In [ ]:
# Create and save the two features
# RF needs one row per match, but pl_team_matches has one row per team appearance. 
# Splitting the rolling residual into home and away columns lets the model compare whetehr each team has been overperforming or underperforming its ELO expectations

home_elo_performance = (
    pl_team_matches[
        pl_team_matches["venue"] == "home"
    ][
        [
            "match_id",
            "elo_performance_last5_pl"
        ]
    ]
    .rename(
        columns={
            "elo_performance_last5_pl":
            "elo_performance_last5_pl_home"
        }
    )
)


away_elo_performance = (
    pl_team_matches[
        pl_team_matches["venue"] == "away"
    ][
        [
            "match_id",
            "elo_performance_last5_pl"
        ]
    ]
    .rename(
        columns={
            "elo_performance_last5_pl":
            "elo_performance_last5_pl_away"
        }
    )
)


elo_performance_features = (
    home_elo_performance
    .merge(
        away_elo_performance,
        on="match_id",
        how="inner"
    )
)


print(
    elo_performance_features.shape
)

print()

print(
    elo_performance_features.head(10)
)

print()

print(
    "Duplicate match IDs:",
    elo_performance_features[
        "match_id"
    ].duplicated().sum()
)

print(
    "Missing home:",
    elo_performance_features[
        "elo_performance_last5_pl_home"
    ].isna().sum()
)

print(
    "Missing away:",
    elo_performance_features[
        "elo_performance_last5_pl_away"
    ].isna().sum()
)


elo_performance_features.to_csv(
    "../data/processed/elo_performance_features.csv",
    index=False
)

print()
print(
    "Saved elo_performance_features.csv"
)

In [ ]:
# ---------------------------------------------------------
# COUNT ONLY PREMIER LEAGUE TEAMS
# ---------------------------------------------------------

pl_european_team_rows = []


for _, row in european_matches.iterrows():

    # Count home team only if it is one of our PL teams
    if row["home_team"] in premier_league_teams:

        pl_european_team_rows.append({
            "team": row["home_team"],
            "season": row["season"],
            "competition": row["competition"],
            "match_id": row["match_id"]
        })


    # Count away team only if it is one of our PL teams
    if row["away_team"] in premier_league_teams:

        pl_european_team_rows.append({
            "team": row["away_team"],
            "season": row["season"],
            "competition": row["competition"],
            "match_id": row["match_id"]
        })


pl_european_team_df = pd.DataFrame(
    pl_european_team_rows
)


# ---------------------------------------------------------
# OVERALL COVERAGE
# ---------------------------------------------------------

team_european_counts = (
    pl_european_team_df[
        "team"
    ]
    .value_counts()
)


print("EUROPEAN MATCHES BY PL TEAM")
print()

print(
    team_european_counts
)


print()

print(
    "PL teams with European matches:",
    pl_european_team_df[
        "team"
    ].nunique()
)

print(
    "Total PL teams:",
    len(premier_league_teams)
)


# ---------------------------------------------------------
# TEAM-SEASON COVERAGE
# ---------------------------------------------------------

print()
print("EUROPEAN MATCHES BY TEAM AND SEASON")
print()


team_season_europe = (
    pl_european_team_df
    .groupby(
        [
            "season",
            "team"
        ]
    )
    .size()
    .reset_index(
        name="european_matches"
    )
)


print(
    team_season_europe.to_string(
        index=False
    )
)


# ---------------------------------------------------------
# NUMBER OF PL CLUBS IN EUROPE EACH SEASON
# ---------------------------------------------------------

print()
print("PL CLUBS WITH EUROPEAN MATCHES BY SEASON")
print()


clubs_per_season = (
    pl_european_team_df
    .groupby(
        "season"
    )["team"]
    .nunique()
)


print(
    clubs_per_season
)

In [ ]:
# ---------------------------------------------------------
# EUROPEAN MATCHES FROM TEAM PERSPECTIVE
# PL CLUBS ONLY
# ---------------------------------------------------------

european_team_rows = []


for _, row in european_matches.iterrows():

    # PL club was the home team
    if row["home_team"] in premier_league_teams:

        european_team_rows.append({
            "match_id": row["match_id"],
            "date": row["date"],
            "season": row["season"],
            "competition": row["competition"],
            "team": row["home_team"],
            "opponent": row["away_team"],
            "venue": "home",
            "goals_for": row["home_goals"],
            "goals_against": row["away_goals"]
        })


    # PL club was the away team
    if row["away_team"] in premier_league_teams:

        european_team_rows.append({
            "match_id": row["match_id"],
            "date": row["date"],
            "season": row["season"],
            "competition": row["competition"],
            "team": row["away_team"],
            "opponent": row["home_team"],
            "venue": "away",
            "goals_for": row["away_goals"],
            "goals_against": row["home_goals"]
        })


european_team_matches = pd.DataFrame(
    european_team_rows
)


european_team_matches["date"] = pd.to_datetime(
    european_team_matches["date"],
    utc=True
)


# ---------------------------------------------------------
# ADD PRE-MATCH ELO
# ---------------------------------------------------------

european_team_matches = (
    european_team_matches
    .merge(
        elo_history_df[
            [
                "match_id",
                "home_elo",
                "away_elo"
            ]
        ],
        on="match_id",
        how="left"
    )
)


european_team_matches[
    "team_elo"
] = np.where(
    european_team_matches["venue"] == "home",
    european_team_matches["home_elo"],
    european_team_matches["away_elo"]
)


european_team_matches[
    "opponent_elo"
] = np.where(
    european_team_matches["venue"] == "home",
    european_team_matches["away_elo"],
    european_team_matches["home_elo"]
)


european_team_matches = (
    european_team_matches
    .sort_values(
        [
            "team",
            "date",
            "match_id"
        ]
    )
    .reset_index(drop=True)
)


print(
    "Shape:",
    european_team_matches.shape
)

print()

print(
    european_team_matches[
        [
            "date",
            "team",
            "opponent",
            "competition",
            "venue",
            "team_elo",
            "opponent_elo",
            "goals_for",
            "goals_against"
        ]
    ].head(20)
)


print()

print(
    "Missing team Elo:",
    european_team_matches[
        "team_elo"
    ].isna().sum()
)

print(
    "Missing opponent Elo:",
    european_team_matches[
        "opponent_elo"
    ].isna().sum()
)

print()

print(
    "Rows:",
    len(european_team_matches)
)

print(
    "Unique PL teams:",
    european_team_matches[
        "team"
    ].nunique()
)

In [ ]:
# Calculate European performance vs ELO expectation

EUROPE_HOME_ADVANTAGE = 40


# ---------------------------------------------------------
# VENUE-ADJUSTED ELO EXPECTATION
# ---------------------------------------------------------

european_team_matches[
    "venue_elo_adjustment"
] = np.where(
    european_team_matches["venue"] == "home",
    EUROPE_HOME_ADVANTAGE,
    -EUROPE_HOME_ADVANTAGE
)


european_team_matches[
    "elo_expected_score"
] = (
    1
    / (
        1
        + 10 ** (
            (
                european_team_matches["opponent_elo"]
                - (
                    european_team_matches["team_elo"]
                    + european_team_matches["venue_elo_adjustment"]
                )
            )
            / 400
        )
    )
)


# ---------------------------------------------------------
# ACTUAL MATCH RESULT
# ---------------------------------------------------------

european_team_matches[
    "actual_score"
] = np.select(
    [
        european_team_matches["goals_for"]
        > european_team_matches["goals_against"],

        european_team_matches["goals_for"]
        == european_team_matches["goals_against"]
    ],
    [
        1.0,
        0.5
    ],
    default=0.0
)


# ---------------------------------------------------------
# PERFORMANCE RELATIVE TO EXPECTATION
# ---------------------------------------------------------

european_team_matches[
    "european_elo_residual"
] = (
    european_team_matches["actual_score"]
    - european_team_matches["elo_expected_score"]
)


# ---------------------------------------------------------
# INSPECT RESULTS
# ---------------------------------------------------------

print(
    european_team_matches[
        [
            "date",
            "team",
            "opponent",
            "venue",
            "team_elo",
            "opponent_elo",
            "elo_expected_score",
            "actual_score",
            "european_elo_residual"
        ]
    ].head(20)
)


print()
print("RESIDUAL SUMMARY")
print()

print(
    european_team_matches[
        "european_elo_residual"
    ].describe()
)


print()
print("AVERAGE BY VENUE")
print()

print(
    european_team_matches
    .groupby("venue")[
        [
            "elo_expected_score",
            "actual_score",
            "european_elo_residual"
        ]
    ]
    .mean()
)

In [ ]:
# ---------------------------------------------------------
# ROLLING EUROPEAN PERFORMANCE
# ---------------------------------------------------------

european_team_matches = (
    european_team_matches
    .sort_values(
        [
            "team",
            "date",
            "match_id"
        ]
    )
    .reset_index(drop=True)
)


european_team_matches[
    "european_performance_last5"
] = (
    european_team_matches
    .groupby("team")[
        "european_elo_residual"
    ]
    .transform(
        lambda x: (
            x.rolling(
                window=5,
                min_periods=1
            )
            .mean()
        )
    )
)


# Number of European matches represented
# in the rolling average
european_team_matches[
    "european_matches_in_window"
] = (
    european_team_matches
    .groupby("team")
    .cumcount()
    .add(1)
    .clip(upper=5)
)


print(
    european_team_matches[
        [
            "date",
            "team",
            "opponent",
            "competition",
            "european_elo_residual",
            "european_performance_last5",
            "european_matches_in_window"
        ]
    ].head(25)
)


print()
print("ROLLING PERFORMANCE SUMMARY")
print()

print(
    european_team_matches[
        "european_performance_last5"
    ].describe()
)


print()
print("WINDOW SIZE COUNTS")
print()

print(
    european_team_matches[
        "european_matches_in_window"
    ].value_counts()
    .sort_index()
)

In [ ]:
# ---------------------------------------------------------
# PREPARE PREMIER LEAGUE TEAM-MATCH ROWS
# ---------------------------------------------------------

pl_for_europe_merge = (
    pl_team_matches[
        [
            "match_id",
            "date",
            "season",
            "team",
            "venue"
        ]
    ]
    .copy()
)


pl_for_europe_merge["date"] = pd.to_datetime(
    pl_for_europe_merge["date"],
    utc=True
)


# ---------------------------------------------------------
# PREPARE EUROPEAN HISTORY
# ---------------------------------------------------------

europe_history = (
    european_team_matches[
        [
            "team",
            "date",
            "european_performance_last5",
            "european_matches_in_window"
        ]
    ]
    .copy()
    .rename(
        columns={
            "date": "last_european_date"
        }
    )
)


# ---------------------------------------------------------
# MERGE EACH TEAM SEPARATELY
#
# This avoids merge_asof sorting problems when multiple
# teams are mixed together.
# ---------------------------------------------------------

merged_team_parts = []


for team_name in sorted(
    pl_for_europe_merge["team"].unique()
):

    team_pl = (
        pl_for_europe_merge[
            pl_for_europe_merge["team"]
            == team_name
        ]
        .copy()
        .sort_values("date")
    )


    team_europe = (
        europe_history[
            europe_history["team"]
            == team_name
        ]
        .copy()
        .sort_values(
            "last_european_date"
        )
    )


    # Teams with no European history
    if team_europe.empty:

        team_pl[
            "last_european_date"
        ] = pd.NaT

        team_pl[
            "european_performance_last5"
        ] = np.nan

        team_pl[
            "european_matches_in_window"
        ] = np.nan


    else:

        team_pl = pd.merge_asof(
            team_pl,
            team_europe[
                [
                    "last_european_date",
                    "european_performance_last5",
                    "european_matches_in_window"
                ]
            ],
            left_on="date",
            right_on="last_european_date",
            direction="backward",
            allow_exact_matches=False
        )


        team_pl["team"] = team_name


    merged_team_parts.append(
        team_pl
    )


pl_europe_features = pd.concat(
    merged_team_parts,
    ignore_index=True
)


# ---------------------------------------------------------
# DAYS SINCE LAST EUROPEAN MATCH
# ---------------------------------------------------------

pl_europe_features[
    "days_since_european_match"
] = (
    (
        pl_europe_features["date"]
        - pl_europe_features["last_european_date"]
    )
    .dt.total_seconds()
    / 86400
)


# ---------------------------------------------------------
# SANITY CHECKS
# ---------------------------------------------------------

print(
    "Shape:",
    pl_europe_features.shape
)

print(
    "Duplicate team-match rows:",
    pl_europe_features[
        [
            "match_id",
            "team"
        ]
    ].duplicated().sum()
)

print()

print(
    "Rows with prior European information:",
    pl_europe_features[
        "european_performance_last5"
    ].notna().sum()
)

print(
    "Percentage with European information:",
    pl_europe_features[
        "european_performance_last5"
    ].notna().mean()
    * 100
)

print()

print(
    "Negative days since Europe:",
    (
        pl_europe_features[
            "days_since_european_match"
        ]
        < 0
    ).sum()
)


print()
print("SAMPLE ARSENAL ROWS")
print()

print(
    pl_europe_features[
        (
            pl_europe_features["team"]
            == "Arsenal"
        )
        &
        (
            pl_europe_features[
                "european_performance_last5"
            ].notna()
        )
    ][
        [
            "date",
            "team",
            "last_european_date",
            "days_since_european_match",
            "european_performance_last5",
            "european_matches_in_window"
        ]
    ]
    .head(20)
    .to_string(index=False)
)

In [ ]:
# ---------------------------------------------------------
# FIX DATE TYPES
# ---------------------------------------------------------

pl_europe_features["date"] = pd.to_datetime(
    pl_europe_features["date"],
    utc=True
)

pl_europe_features["last_european_date"] = pd.to_datetime(
    pl_europe_features["last_european_date"],
    utc=True,
    errors="coerce"
)


# ---------------------------------------------------------
# DAYS SINCE LAST EUROPEAN MATCH
# ---------------------------------------------------------

pl_europe_features[
    "days_since_european_match"
] = (
    (
        pl_europe_features["date"]
        - pl_europe_features["last_european_date"]
    )
    .dt.total_seconds()
    / 86400
)


# ---------------------------------------------------------
# SANITY CHECKS
# ---------------------------------------------------------

print(
    "Shape:",
    pl_europe_features.shape
)

print(
    "Duplicate team-match rows:",
    pl_europe_features[
        [
            "match_id",
            "team"
        ]
    ].duplicated().sum()
)

print()

print(
    "Rows with prior European information:",
    pl_europe_features[
        "european_performance_last5"
    ].notna().sum()
)

print(
    "Percentage with European information:",
    pl_europe_features[
        "european_performance_last5"
    ].notna().mean()
    * 100
)

print()

print(
    "Negative days since Europe:",
    (
        pl_europe_features[
            "days_since_european_match"
        ]
        < 0
    ).sum()
)


print()
print("SAMPLE ARSENAL ROWS")
print()

print(
    pl_europe_features[
        (
            pl_europe_features["team"]
            == "Arsenal"
        )
        &
        (
            pl_europe_features[
                "european_performance_last5"
            ].notna()
        )
    ][
        [
            "date",
            "team",
            "last_european_date",
            "days_since_european_match",
            "european_performance_last5",
            "european_matches_in_window"
        ]
    ]
    .head(20)
    .to_string(index=False)
)

In [ ]:
# ---------------------------------------------------------
# EUROPEAN FEATURE FRESHNESS
# ---------------------------------------------------------

freshness = (
    pl_europe_features[
        pl_europe_features[
            "days_since_european_match"
        ].notna()
    ][
        "days_since_european_match"
    ]
)


print("FRESHNESS SUMMARY")
print()

print(
    freshness.describe(
        percentiles=[
            0.25,
            0.50,
            0.75,
            0.80,
            0.90,
            0.95
        ]
    )
)


print()
print("COVERAGE AT DIFFERENT FRESHNESS CUTOFFS")
print()


for cutoff in [
    7,
    14,
    21,
    30,
    45,
    60,
    90,
    180
]:

    fresh_rows = (
        pl_europe_features[
            "days_since_european_match"
        ]
        <= cutoff
    ).sum()

    percentage = (
        fresh_rows
        / len(pl_europe_features)
        * 100
    )

    print(
        f"{cutoff:>3} days: "
        f"{fresh_rows:>4} rows "
        f"({percentage:.2f}%)"
    )


# ---------------------------------------------------------
# FRESHNESS BY SEASON
# Using 30 days only as a diagnostic for now
# ---------------------------------------------------------

print()
print("30-DAY COVERAGE BY SEASON")
print()


coverage_30 = (
    pl_europe_features
    .assign(
        fresh_europe_30=(
            pl_europe_features[
                "days_since_european_match"
            ]
            <= 30
        )
    )
    .groupby("season")[
        "fresh_europe_30"
    ]
    .agg(
        [
            "sum",
            "mean"
        ]
    )
)


coverage_30["mean"] = (
    coverage_30["mean"]
    * 100
)


print(
    coverage_30.rename(
        columns={
            "sum": "rows",
            "mean": "percentage"
        }
    )
)

In [ ]:
EUROPE_FRESHNESS_DAYS = 30


# ---------------------------------------------------------
# CREATE FRESH EUROPEAN FEATURES
# ---------------------------------------------------------

pl_europe_features[
    "recent_europe"
] = (
    pl_europe_features[
        "days_since_european_match"
    ]
    <= EUROPE_FRESHNESS_DAYS
).astype(int)


pl_europe_features[
    "european_performance_recent"
] = np.where(
    pl_europe_features[
        "recent_europe"
    ] == 1,
    pl_europe_features[
        "european_performance_last5"
    ],
    np.nan
)


# ---------------------------------------------------------
# SPLIT HOME AND AWAY
# ---------------------------------------------------------

europe_home = (
    pl_europe_features[
        pl_europe_features["venue"]
        == "home"
    ][
        [
            "match_id",
            "recent_europe",
            "european_performance_recent",
            "days_since_european_match"
        ]
    ]
    .rename(
        columns={
            "recent_europe":
                "recent_europe_home",

            "european_performance_recent":
                "european_performance_recent_home",

            "days_since_european_match":
                "days_since_european_match_home"
        }
    )
)


europe_away = (
    pl_europe_features[
        pl_europe_features["venue"]
        == "away"
    ][
        [
            "match_id",
            "recent_europe",
            "european_performance_recent",
            "days_since_european_match"
        ]
    ]
    .rename(
        columns={
            "recent_europe":
                "recent_europe_away",

            "european_performance_recent":
                "european_performance_recent_away",

            "days_since_european_match":
                "days_since_european_match_away"
        }
    )
)


# ---------------------------------------------------------
# COMBINE INTO ONE ROW PER PL MATCH
# ---------------------------------------------------------

europe_match_features = (
    europe_home
    .merge(
        europe_away,
        on="match_id",
        how="inner"
    )
)


print(
    "Shape:",
    europe_match_features.shape
)

print(
    "Duplicate match IDs:",
    europe_match_features[
        "match_id"
    ].duplicated().sum()
)


print()
print("RECENT EUROPE COUNTS")
print()

print(
    "Home team:",
    europe_match_features[
        "recent_europe_home"
    ].sum()
)

print(
    "Away team:",
    europe_match_features[
        "recent_europe_away"
    ].sum()
)


print()
print("MATCH COVERAGE")
print()

at_least_one = (
    (
        europe_match_features[
            "recent_europe_home"
        ] == 1
    )
    |
    (
        europe_match_features[
            "recent_europe_away"
        ] == 1
    )
)

both = (
    (
        europe_match_features[
            "recent_europe_home"
        ] == 1
    )
    &
    (
        europe_match_features[
            "recent_europe_away"
        ] == 1
    )
)


print(
    "At least one team recently in Europe:",
    at_least_one.sum(),
    f"({at_least_one.mean() * 100:.2f}%)"
)

print(
    "Both teams recently in Europe:",
    both.sum(),
    f"({both.mean() * 100:.2f}%)"
)


print()
print("MISSING RECENT PERFORMANCE")
print()

print(
    europe_match_features[
        [
            "european_performance_recent_home",
            "european_performance_recent_away"
        ]
    ]
    .isna()
    .sum()
)


# ---------------------------------------------------------
# SAVE
# ---------------------------------------------------------

europe_match_features.to_csv(
    "../data/processed/european_performance_features.csv",
    index=False
)

print()
print(
    "Saved:",
    "../data/processed/european_performance_features.csv"
)

In [ ]:
# Look beyond raw pre match statistics

# ---------------------------------------------------------
# BUILD ADDITIONAL MATCH-DOMINANCE DIFFERENCES
# ---------------------------------------------------------

team_matches[
    "shot_difference"
] = (
    team_matches["shots_for"]
    - team_matches["shots_against"]
)


team_matches[
    "possession_difference"
] = (
    team_matches["possession_for"]
    - team_matches["possession_against"]
)


team_matches[
    "corner_difference"
] = (
    team_matches["corners_for"]
    - team_matches["corners_against"]
)


# ---------------------------------------------------------
# LEAKAGE-SAFE ROLLING LAST 5
# ---------------------------------------------------------

for stat in [
    "shot_difference",
    "possession_difference",
    "corner_difference"
]:

    team_matches[
        f"{stat}_last5_all"
    ] = (
        team_matches
        .groupby("team")[stat]
        .transform(
            lambda x: (
                x.shift(1)
                .rolling(
                    window=5,
                    min_periods=1
                )
                .mean()
            )
        )
    )


# ---------------------------------------------------------
# PREMIER LEAGUE ROWS ONLY
# ---------------------------------------------------------

dominance_check = (
    team_matches[
        team_matches["competition"]
        == "premier_league"
    ][
        [
            "match_id",
            "date",
            "season",
            "team",
            "venue",
            "points",
            "goal_difference",
            "shot_difference_last5_all",
            "possession_difference_last5_all",
            "corner_difference_last5_all"
        ]
    ]
    .copy()
)


print(
    "Shape:",
    dominance_check.shape
)


print()
print("MISSING VALUES")
print()

print(
    dominance_check[
        [
            "shot_difference_last5_all",
            "possession_difference_last5_all",
            "corner_difference_last5_all"
        ]
    ]
    .isna()
    .sum()
)


print()
print("CORRELATION WITH NEXT-MATCH PERFORMANCE")
print()

print(
    dominance_check[
        [
            "points",
            "goal_difference",
            "shot_difference_last5_all",
            "possession_difference_last5_all",
            "corner_difference_last5_all"
        ]
    ]
    .corr()
    .round(3)
)


print()
print("FEATURE SUMMARY")
print()

print(
    dominance_check[
        [
            "shot_difference_last5_all",
            "possession_difference_last5_all",
            "corner_difference_last5_all"
        ]
    ]
    .describe()
)

In [ ]:
# Test these signals together and individually

# ---------------------------------------------------------
# KEEP PREMIER LEAGUE DOMINANCE FEATURES
# ---------------------------------------------------------

dominance_team_features = (
    dominance_check[
        [
            "match_id",
            "team",
            "venue",
            "shot_difference_last5_all",
            "possession_difference_last5_all",
            "corner_difference_last5_all"
        ]
    ]
    .copy()
)


# ---------------------------------------------------------
# HOME FEATURES
# ---------------------------------------------------------

dominance_home = (
    dominance_team_features[
        dominance_team_features["venue"]
        == "home"
    ][
        [
            "match_id",
            "shot_difference_last5_all",
            "possession_difference_last5_all",
            "corner_difference_last5_all"
        ]
    ]
    .rename(
        columns={
            "shot_difference_last5_all":
                "shot_difference_last5_all_home",

            "possession_difference_last5_all":
                "possession_difference_last5_all_home",

            "corner_difference_last5_all":
                "corner_difference_last5_all_home"
        }
    )
)


# ---------------------------------------------------------
# AWAY FEATURES
# ---------------------------------------------------------

dominance_away = (
    dominance_team_features[
        dominance_team_features["venue"]
        == "away"
    ][
        [
            "match_id",
            "shot_difference_last5_all",
            "possession_difference_last5_all",
            "corner_difference_last5_all"
        ]
    ]
    .rename(
        columns={
            "shot_difference_last5_all":
                "shot_difference_last5_all_away",

            "possession_difference_last5_all":
                "possession_difference_last5_all_away",

            "corner_difference_last5_all":
                "corner_difference_last5_all_away"
        }
    )
)


# ---------------------------------------------------------
# ONE ROW PER PREMIER LEAGUE MATCH
# ---------------------------------------------------------

dominance_match_features = (
    dominance_home
    .merge(
        dominance_away,
        on="match_id",
        how="inner"
    )
)


print(
    "Shape:",
    dominance_match_features.shape
)

print(
    "Duplicate match IDs:",
    dominance_match_features[
        "match_id"
    ].duplicated().sum()
)


print()
print("MISSING VALUES")
print()

print(
    dominance_match_features
    .drop(
        columns="match_id"
    )
    .isna()
    .sum()
)


print()
print("SAMPLE")
print()

print(
    dominance_match_features
    .head(10)
    .to_string(index=False)
)


# ---------------------------------------------------------
# SAVE
# ---------------------------------------------------------

dominance_match_features.to_csv(
    "../data/processed/dominance_features.csv",
    index=False
)

print()
print(
    "Saved:",
    "../data/processed/dominance_features.csv"
)


In [ ]:
import pandas as pd
import numpy as np


# ---------------------------------------------------------
# LOAD UPDATED HISTORICAL MATCH DATA
# ---------------------------------------------------------

historical_df = pd.read_csv(
    "../data/processed/historical_matches.csv"
)


historical_df["date"] = pd.to_datetime(
    historical_df["date"],
    utc=True,
    errors="coerce"
)


historical_df = (
    historical_df
    .sort_values(
        [
            "date",
            "match_id"
        ]
    )
    .reset_index(
        drop=True
    )
)


print(
    "Historical shape:",
    historical_df.shape
)


print()

print(
    "Premier League matches:",
    (
        historical_df[
            "competition"
        ]
        == "premier_league"
    ).sum()
)


print()

print(
    "2026/27 Premier League matches:",
    (
        (
            historical_df[
                "competition"
            ]
            == "premier_league"
        )
        &
        (
            historical_df[
                "season"
            ]
            == "2026_27"
        )
    ).sum()
)

In [ ]:
# ---------------------------------------------------------
# BUILD TEAM-PERSPECTIVE MATCH TABLE
# ---------------------------------------------------------

home_rows = pd.DataFrame({

    "match_id":
        historical_df["match_id"],

    "date":
        historical_df["date"],

    "season":
        historical_df["season"],

    "competition":
        historical_df["competition"],

    "team":
        historical_df["home_team"],

    "opponent":
        historical_df["away_team"],

    "venue":
        "home",

    "goals_for":
        historical_df["home_goals"],

    "goals_against":
        historical_df["away_goals"],

    "xg_for":
        historical_df["home_xg"],

    "xg_against":
        historical_df["away_xg"],

    "shots_for":
        historical_df["home_shots"],

    "shots_against":
        historical_df["away_shots"],

    "shots_on_target_for":
        historical_df["home_shots_on_target"],

    "shots_on_target_against":
        historical_df["away_shots_on_target"],

    "possession_for":
        historical_df["home_possession"],

    "possession_against":
        historical_df["away_possession"],

    "passes_for":
        historical_df["home_passes"],

    "passes_against":
        historical_df["away_passes"],

    "corners_for":
        historical_df["home_corners"],

    "corners_against":
        historical_df["away_corners"]
})


away_rows = pd.DataFrame({

    "match_id":
        historical_df["match_id"],

    "date":
        historical_df["date"],

    "season":
        historical_df["season"],

    "competition":
        historical_df["competition"],

    "team":
        historical_df["away_team"],

    "opponent":
        historical_df["home_team"],

    "venue":
        "away",

    "goals_for":
        historical_df["away_goals"],

    "goals_against":
        historical_df["home_goals"],

    "xg_for":
        historical_df["away_xg"],

    "xg_against":
        historical_df["home_xg"],

    "shots_for":
        historical_df["away_shots"],

    "shots_against":
        historical_df["home_shots"],

    "shots_on_target_for":
        historical_df["away_shots_on_target"],

    "shots_on_target_against":
        historical_df["home_shots_on_target"],

    "possession_for":
        historical_df["away_possession"],

    "possession_against":
        historical_df["home_possession"],

    "passes_for":
        historical_df["away_passes"],

    "passes_against":
        historical_df["home_passes"],

    "corners_for":
        historical_df["away_corners"],

    "corners_against":
        historical_df["home_corners"]
})


team_matches = pd.concat(
    [
        home_rows,
        away_rows
    ],
    ignore_index=True
)


team_matches = (
    team_matches
    .sort_values(
        [
            "team",
            "date",
            "match_id"
        ]
    )
    .reset_index(
        drop=True
    )
)


print(
    "Team matches shape:",
    team_matches.shape
)


print()

print(
    "2026/27 PL team rows:",
    (
        (
            team_matches["competition"]
            == "premier_league"
        )
        &
        (
            team_matches["season"]
            == "2026_27"
        )
    ).sum()
)

In [ ]:
# ---------------------------------------------------------
# BASIC TEAM-MATCH FEATURES
# ---------------------------------------------------------

# Result from the team's perspective
team_matches["result"] = np.select(
    [
        team_matches["goals_for"] > team_matches["goals_against"],
        team_matches["goals_for"] < team_matches["goals_against"]
    ],
    [
        "W",
        "L"
    ],
    default="D"
)


# Points earned
team_matches["points"] = np.select(
    [
        team_matches["result"] == "W",
        team_matches["result"] == "D"
    ],
    [
        3,
        1
    ],
    default=0
)


# Goal difference
team_matches["goal_difference"] = (
    team_matches["goals_for"]
    - team_matches["goals_against"]
)


# ---------------------------------------------------------
# DAYS REST
# ---------------------------------------------------------

# team_matches is already sorted by:
# team -> date -> match_id

team_matches["previous_match_date"] = (
    team_matches
    .groupby("team")["date"]
    .shift(1)
)


team_matches["days_rest"] = (
    team_matches["date"]
    - team_matches["previous_match_date"]
).dt.total_seconds() / 86400


# ---------------------------------------------------------
# CHECK
# ---------------------------------------------------------

print(
    "Team matches shape:",
    team_matches.shape
)

print()

print(
    team_matches[
        (
            team_matches["competition"] == "premier_league"
        )
        &
        (
            team_matches["season"] == "2026_27"
        )
    ][
        [
            "date",
            "team",
            "opponent",
            "venue",
            "result",
            "points",
            "goal_difference",
            "days_rest"
        ]
    ]
    .sort_values("date")
    .tail(10)
)

In [ ]:
# ---------------------------------------------------------
# RELOAD CORRECTED HISTORICAL DATA
# ---------------------------------------------------------

import pandas as pd
import numpy as np


historical_df = pd.read_csv(
    "../data/processed/historical_matches.csv"
)


historical_df["date"] = pd.to_datetime(
    historical_df["date"],
    utc=True,
    errors="coerce"
)


historical_df = (
    historical_df
    .sort_values(
        [
            "date",
            "match_id"
        ]
    )
    .reset_index(
        drop=True
    )
)


# ---------------------------------------------------------
# CHECK CURRENT PREMIER LEAGUE SEASON
# ---------------------------------------------------------

current_pl = historical_df[
    (
        historical_df["competition"]
        == "premier_league"
    )
    &
    (
        historical_df["season"]
        == "2026_27"
    )
].copy()


print(
    "Historical shape:",
    historical_df.shape
)

print(
    "2026/27 PL matches:",
    len(current_pl)
)

print(
    "Missing 2026/27 dates:",
    current_pl["date"].isna().sum()
)

print(
    "Missing 2026/27 scores:",
    current_pl[
        [
            "home_goals",
            "away_goals"
        ]
    ].isna().any(axis=1).sum()
)

print()

print(
    current_pl[
        [
            "date",
            "home_team",
            "away_team",
            "home_goals",
            "away_goals"
        ]
    ]
    .sort_values("date")
    .tail(5)
    .to_string(index=False)
)

In [ ]:
# ---------------------------------------------------------
# REBUILD TEAM-PERSPECTIVE MATCH TABLE
# ---------------------------------------------------------

home_rows = pd.DataFrame({

    "match_id": historical_df["match_id"],
    "date": historical_df["date"],
    "season": historical_df["season"],
    "competition": historical_df["competition"],

    "team": historical_df["home_team"],
    "opponent": historical_df["away_team"],
    "venue": "home",

    "goals_for": historical_df["home_goals"],
    "goals_against": historical_df["away_goals"],

    "xg_for": historical_df["home_xg"],
    "xg_against": historical_df["away_xg"],

    "shots_for": historical_df["home_shots"],
    "shots_against": historical_df["away_shots"],

    "shots_on_target_for":
        historical_df["home_shots_on_target"],

    "shots_on_target_against":
        historical_df["away_shots_on_target"],

    "possession_for":
        historical_df["home_possession"],

    "possession_against":
        historical_df["away_possession"],

    "passes_for":
        historical_df["home_passes"],

    "passes_against":
        historical_df["away_passes"],

    "corners_for":
        historical_df["home_corners"],

    "corners_against":
        historical_df["away_corners"]
})


away_rows = pd.DataFrame({

    "match_id": historical_df["match_id"],
    "date": historical_df["date"],
    "season": historical_df["season"],
    "competition": historical_df["competition"],

    "team": historical_df["away_team"],
    "opponent": historical_df["home_team"],
    "venue": "away",

    "goals_for": historical_df["away_goals"],
    "goals_against": historical_df["home_goals"],

    "xg_for": historical_df["away_xg"],
    "xg_against": historical_df["home_xg"],

    "shots_for": historical_df["away_shots"],
    "shots_against": historical_df["home_shots"],

    "shots_on_target_for":
        historical_df["away_shots_on_target"],

    "shots_on_target_against":
        historical_df["home_shots_on_target"],

    "possession_for":
        historical_df["away_possession"],

    "possession_against":
        historical_df["home_possession"],

    "passes_for":
        historical_df["away_passes"],

    "passes_against":
        historical_df["home_passes"],

    "corners_for":
        historical_df["away_corners"],

    "corners_against":
        historical_df["home_corners"]
})


team_matches = pd.concat(
    [
        home_rows,
        away_rows
    ],
    ignore_index=True
)


team_matches = (
    team_matches
    .sort_values(
        [
            "team",
            "date",
            "match_id"
        ]
    )
    .reset_index(drop=True)
)


# ---------------------------------------------------------
# CHECK
# ---------------------------------------------------------

current_team_rows = team_matches[
    (
        team_matches["competition"]
        == "premier_league"
    )
    &
    (
        team_matches["season"]
        == "2026_27"
    )
]


print(
    "Team matches shape:",
    team_matches.shape
)

print(
    "2026/27 PL team rows:",
    len(current_team_rows)
)

print(
    "Missing 2026/27 dates:",
    current_team_rows["date"].isna().sum()
)

print()

print(
    current_team_rows[
        [
            "date",
            "team",
            "opponent",
            "venue",
            "goals_for",
            "goals_against"
        ]
    ]
    .sort_values("date")
    .tail(10)
    .to_string(index=False)
)

In [ ]:
# ---------------------------------------------------------
# BASIC TEAM-MATCH FEATURES
# ---------------------------------------------------------

team_matches["result"] = np.select(
    [
        team_matches["goals_for"] > team_matches["goals_against"],
        team_matches["goals_for"] < team_matches["goals_against"]
    ],
    [
        "W",
        "L"
    ],
    default="D"
)


team_matches["points"] = np.select(
    [
        team_matches["result"] == "W",
        team_matches["result"] == "D"
    ],
    [
        3,
        1
    ],
    default=0
)


team_matches["goal_difference"] = (
    team_matches["goals_for"]
    - team_matches["goals_against"]
)


# ---------------------------------------------------------
# DAYS REST
# ---------------------------------------------------------

team_matches["previous_match_date"] = (
    team_matches
    .groupby("team")["date"]
    .shift(1)
)


team_matches["days_rest"] = (
    team_matches["date"]
    - team_matches["previous_match_date"]
).dt.total_seconds() / 86400


# ---------------------------------------------------------
# CHECK 2026/27 PREMIER LEAGUE ROWS
# ---------------------------------------------------------

check_rest = (
    team_matches[
        (
            team_matches["competition"] == "premier_league"
        )
        &
        (
            team_matches["season"] == "2026_27"
        )
    ][
        [
            "date",
            "team",
            "opponent",
            "result",
            "points",
            "goal_difference",
            "days_rest"
        ]
    ]
    .sort_values(
        [
            "date",
            "team"
        ]
    )
)


print(
    "Team matches shape:",
    team_matches.shape
)

print()

print(
    "Missing 2026/27 rest values:",
    check_rest["days_rest"].isna().sum()
)

print(
    "Negative 2026/27 rest values:",
    (check_rest["days_rest"] < 0).sum()
)

print()

print(
    check_rest.tail(12).to_string(index=False)
)

In [ ]:
# ---------------------------------------------------------
# ALL-COMPETITION ROLLING FEATURES
# ---------------------------------------------------------

# Win indicator
team_matches["win"] = (
    team_matches["result"] == "W"
).astype(int)


# Match-level performance differences
team_matches["xg_difference"] = (
    team_matches["xg_for"]
    - team_matches["xg_against"]
)


team_matches["shots_on_target_difference"] = (
    team_matches["shots_on_target_for"]
    - team_matches["shots_on_target_against"]
)


# ---------------------------------------------------------
# LEAKAGE-SAFE LAST-5 ALL-COMP FEATURES
# ---------------------------------------------------------

all_comp_rolling_columns = {

    "points":
        "points_last5_all",

    "goal_difference":
        "goal_difference_last5_all",

    "win":
        "win_rate_last5_all",

    "xg_difference":
        "xg_difference_last5_all",

    "shots_on_target_difference":
        "shots_on_target_difference_last5_all"
}


for source_col, new_col in all_comp_rolling_columns.items():

    team_matches[new_col] = (
        team_matches
        .groupby("team")[source_col]
        .transform(
            lambda x:
                x.shift(1)
                .rolling(
                    window=5,
                    min_periods=1
                )
                .mean()
        )
    )


# ---------------------------------------------------------
# CHECK CURRENT PREMIER LEAGUE SEASON
# ---------------------------------------------------------

check_all_form = (
    team_matches[
        (
            team_matches["competition"]
            == "premier_league"
        )
        &
        (
            team_matches["season"]
            == "2026_27"
        )
    ][
        [
            "date",
            "team",
            "opponent",
            "points_last5_all",
            "goal_difference_last5_all",
            "win_rate_last5_all",
            "shots_on_target_difference_last5_all"
        ]
    ]
    .sort_values(
        [
            "date",
            "team"
        ]
    )
)


print(
    "Team matches shape:",
    team_matches.shape
)

print()

print(
    "2026/27 PL rows:",
    len(check_all_form)
)

print()

print(
    "Missing points_last5_all:",
    check_all_form["points_last5_all"].isna().sum()
)

print(
    "Missing goal_difference_last5_all:",
    check_all_form["goal_difference_last5_all"].isna().sum()
)

print(
    "Missing win_rate_last5_all:",
    check_all_form["win_rate_last5_all"].isna().sum()
)

print()

print(
    check_all_form.tail(12).to_string(index=False)
)

In [ ]:
# ---------------------------------------------------------
# PREMIER-LEAGUE-ONLY ROLLING FEATURES
# ---------------------------------------------------------

pl_team_matches = (
    team_matches[
        team_matches["competition"] == "premier_league"
    ]
    .copy()
)


pl_team_matches = (
    pl_team_matches
    .sort_values(
        [
            "team",
            "date",
            "match_id"
        ]
    )
    .reset_index(drop=True)
)


# ---------------------------------------------------------
# LAST-5 PL FEATURES
# ---------------------------------------------------------

pl_last5_columns = {

    "points":
        "points_last5_pl",

    "goal_difference":
        "goal_difference_last5_pl",

    "win":
        "win_rate_last5_pl",

    "xg_difference":
        "xg_difference_last5_pl",

    "shots_on_target_difference":
        "shots_on_target_difference_last5_pl"
}


for source_col, new_col in pl_last5_columns.items():

    pl_team_matches[new_col] = (
        pl_team_matches
        .groupby("team")[source_col]
        .transform(
            lambda x:
                x.shift(1)
                .rolling(
                    window=5,
                    min_periods=1
                )
                .mean()
        )
    )


# ---------------------------------------------------------
# LAST-10 PL FEATURES
# ---------------------------------------------------------

pl_last10_columns = {

    "points":
        "points_last10_pl",

    "goal_difference":
        "goal_difference_last10_pl",

    "win":
        "win_rate_last10_pl"
}


for source_col, new_col in pl_last10_columns.items():

    pl_team_matches[new_col] = (
        pl_team_matches
        .groupby("team")[source_col]
        .transform(
            lambda x:
                x.shift(1)
                .rolling(
                    window=10,
                    min_periods=1
                )
                .mean()
        )
    )


# ---------------------------------------------------------
# CHECK
# ---------------------------------------------------------

check_pl_form = (
    pl_team_matches[
        pl_team_matches["season"] == "2026_27"
    ][
        [
            "date",
            "team",
            "opponent",
            "points_last5_pl",
            "goal_difference_last5_pl",
            "win_rate_last5_pl",
            "shots_on_target_difference_last5_pl",
            "points_last10_pl",
            "goal_difference_last10_pl",
            "win_rate_last10_pl"
        ]
    ]
    .sort_values(
        [
            "date",
            "team"
        ]
    )
)


print(
    "PL team matches shape:",
    pl_team_matches.shape
)

print()

print(
    "2026/27 PL team rows:",
    len(check_pl_form)
)

print()

print(
    "Missing points_last5_pl:",
    check_pl_form["points_last5_pl"].isna().sum()
)

print(
    "Missing points_last10_pl:",
    check_pl_form["points_last10_pl"].isna().sum()
)

print()

print(
    check_pl_form.tail(12).to_string(index=False)
)

In [ ]:
# ---------------------------------------------------------
# RELOAD CORRECTED HISTORICAL DATA
# ---------------------------------------------------------

import pandas as pd
import numpy as np


historical_df = pd.read_csv(
    "../data/processed/historical_matches.csv"
)


historical_df["date"] = pd.to_datetime(
    historical_df["date"],
    utc=True,
    errors="coerce"
)


historical_df = (
    historical_df
    .sort_values(
        [
            "date",
            "match_id"
        ]
    )
    .reset_index(drop=True)
)


print(
    "Historical shape:",
    historical_df.shape
)


# ---------------------------------------------------------
# BUILD TEAM-PERSPECTIVE MATCH DATA
# ---------------------------------------------------------

home_rows = pd.DataFrame({

    "match_id":
        historical_df["match_id"],

    "date":
        historical_df["date"],

    "season":
        historical_df["season"],

    "competition":
        historical_df["competition"],

    "team":
        historical_df["home_team"],

    "opponent":
        historical_df["away_team"],

    "venue":
        "home",

    "goals_for":
        historical_df["home_goals"],

    "goals_against":
        historical_df["away_goals"],

    "xg_for":
        historical_df["home_xg"],

    "xg_against":
        historical_df["away_xg"],

    "shots_for":
        historical_df["home_shots"],

    "shots_against":
        historical_df["away_shots"],

    "shots_on_target_for":
        historical_df["home_shots_on_target"],

    "shots_on_target_against":
        historical_df["away_shots_on_target"],

    "possession_for":
        historical_df["home_possession"],

    "possession_against":
        historical_df["away_possession"],

    "passes_for":
        historical_df["home_passes"],

    "passes_against":
        historical_df["away_passes"],

    "corners_for":
        historical_df["home_corners"],

    "corners_against":
        historical_df["away_corners"]
})


away_rows = pd.DataFrame({

    "match_id":
        historical_df["match_id"],

    "date":
        historical_df["date"],

    "season":
        historical_df["season"],

    "competition":
        historical_df["competition"],

    "team":
        historical_df["away_team"],

    "opponent":
        historical_df["home_team"],

    "venue":
        "away",

    "goals_for":
        historical_df["away_goals"],

    "goals_against":
        historical_df["home_goals"],

    "xg_for":
        historical_df["away_xg"],

    "xg_against":
        historical_df["home_xg"],

    "shots_for":
        historical_df["away_shots"],

    "shots_against":
        historical_df["home_shots"],

    "shots_on_target_for":
        historical_df["away_shots_on_target"],

    "shots_on_target_against":
        historical_df["home_shots_on_target"],

    "possession_for":
        historical_df["away_possession"],

    "possession_against":
        historical_df["home_possession"],

    "passes_for":
        historical_df["away_passes"],

    "passes_against":
        historical_df["home_passes"],

    "corners_for":
        historical_df["away_corners"],

    "corners_against":
        historical_df["home_corners"]
})


team_matches = pd.concat(
    [
        home_rows,
        away_rows
    ],
    ignore_index=True
)


team_matches = (
    team_matches
    .sort_values(
        [
            "team",
            "date",
            "match_id"
        ]
    )
    .reset_index(drop=True)
)


print(
    "Team matches shape:",
    team_matches.shape
)


# ---------------------------------------------------------
# BASIC FEATURES
# ---------------------------------------------------------

team_matches["result"] = np.select(

    [
        team_matches["goals_for"]
        > team_matches["goals_against"],

        team_matches["goals_for"]
        < team_matches["goals_against"]
    ],

    [
        "W",
        "L"
    ],

    default="D"
)


team_matches["points"] = np.select(

    [
        team_matches["result"] == "W",
        team_matches["result"] == "D"
    ],

    [
        3,
        1
    ],

    default=0
)


team_matches["goal_difference"] = (
    team_matches["goals_for"]
    -
    team_matches["goals_against"]
)


team_matches["previous_match_date"] = (
    team_matches
    .groupby("team")["date"]
    .shift(1)
)


team_matches["days_rest"] = (

    team_matches["date"]
    -
    team_matches["previous_match_date"]

).dt.days


team_matches["win"] = (
    team_matches["result"] == "W"
).astype(int)


team_matches["xg_difference"] = (
    team_matches["xg_for"]
    -
    team_matches["xg_against"]
)


team_matches["shots_on_target_difference"] = (
    team_matches["shots_on_target_for"]
    -
    team_matches["shots_on_target_against"]
)


# ---------------------------------------------------------
# ALL-COMPETITION LAST-5 FEATURES
# ---------------------------------------------------------

all_comp_last5 = {

    "points":
        "points_last5_all",

    "goal_difference":
        "goal_difference_last5_all",

    "win":
        "win_rate_last5_all",

    "xg_difference":
        "xg_difference_last5_all",

    "shots_on_target_difference":
        "shots_on_target_difference_last5_all"
}


for source_col, new_col in all_comp_last5.items():

    team_matches[new_col] = (
        team_matches
        .groupby("team")[source_col]
        .transform(
            lambda x:
                x.shift(1)
                .rolling(
                    window=5,
                    min_periods=1
                )
                .mean()
        )
    )


# ---------------------------------------------------------
# PREMIER LEAGUE ONLY DATA
# ---------------------------------------------------------

pl_team_matches = (
    team_matches[
        team_matches["competition"]
        == "premier_league"
    ]
    .copy()
)


pl_team_matches = (
    pl_team_matches
    .sort_values(
        [
            "team",
            "date",
            "match_id"
        ]
    )
    .reset_index(drop=True)
)


# ---------------------------------------------------------
# PL LAST-5 FEATURES
# ---------------------------------------------------------

pl_last5_columns = {

    "points":
        "points_last5_pl",

    "goal_difference":
        "goal_difference_last5_pl",

    "win":
        "win_rate_last5_pl",

    "xg_difference":
        "xg_difference_last5_pl",

    "shots_on_target_difference":
        "shots_on_target_difference_last5_pl"
}


for source_col, new_col in pl_last5_columns.items():

    pl_team_matches[new_col] = (
        pl_team_matches
        .groupby("team")[source_col]
        .transform(
            lambda x:
                x.shift(1)
                .rolling(
                    window=5,
                    min_periods=1
                )
                .mean()
        )
    )


# ---------------------------------------------------------
# PL LAST-10 FEATURES
# ---------------------------------------------------------

pl_last10_columns = {

    "points":
        "points_last10_pl",

    "goal_difference":
        "goal_difference_last10_pl",

    "win":
        "win_rate_last10_pl"
}


for source_col, new_col in pl_last10_columns.items():

    pl_team_matches[new_col] = (
        pl_team_matches
        .groupby("team")[source_col]
        .transform(
            lambda x:
                x.shift(1)
                .rolling(
                    window=10,
                    min_periods=1
                )
                .mean()
        )
    )


# ---------------------------------------------------------
# QUICK CHECK
# ---------------------------------------------------------

check_2026 = (
    pl_team_matches[
        pl_team_matches["season"]
        == "2026_27"
    ]
)


print()

print(
    "PL team matches shape:",
    pl_team_matches.shape
)

print(
    "2026/27 rows:",
    len(check_2026)
)

print()

print(
    "Missing SOT difference:",
    check_2026[
        "shots_on_target_difference"
    ].isna().sum()
)

print(
    "Missing SOT last5 all:",
    check_2026[
        "shots_on_target_difference_last5_all"
    ].isna().sum()
)

print(
    "Missing SOT last5 PL:",
    check_2026[
        "shots_on_target_difference_last5_pl"
    ].isna().sum()
)

In [ ]:
# ---------------------------------------------------------
# MATCH CONGESTION FEATURES
# ALL COMPETITIONS
# ---------------------------------------------------------

import numpy as np
import pandas as pd


# Work from all team matches because cup/European matches
# also contribute to fixture congestion.
congestion_df = (
    team_matches[
        [
            "match_id",
            "team",
            "date"
        ]
    ]
    .copy()
    .sort_values(
        [
            "team",
            "date",
            "match_id"
        ]
    )
    .reset_index(drop=True)
)


congestion_df["matches_last_7_days"] = 0
congestion_df["matches_last_14_days"] = 0


# ---------------------------------------------------------
# COUNT PRIOR MATCHES ONLY
# ---------------------------------------------------------

for team, group in congestion_df.groupby(
    "team",
    sort=False
):

    group = group.sort_values(
        [
            "date",
            "match_id"
        ]
    )

    dates = group["date"].values.astype(
        "datetime64[ns]"
    )


    counts_7 = []
    counts_14 = []


    for i, current_date in enumerate(dates):

        previous_dates = dates[:i]


        # Previous 7 days
        cutoff_7 = (
            current_date
            -
            np.timedelta64(7, "D")
        )

        count_7 = np.sum(
            previous_dates >= cutoff_7
        )


        # Previous 14 days
        cutoff_14 = (
            current_date
            -
            np.timedelta64(14, "D")
        )

        count_14 = np.sum(
            previous_dates >= cutoff_14
        )


        counts_7.append(
            count_7
        )

        counts_14.append(
            count_14
        )


    congestion_df.loc[
        group.index,
        "matches_last_7_days"
    ] = counts_7


    congestion_df.loc[
        group.index,
        "matches_last_14_days"
    ] = counts_14


# ---------------------------------------------------------
# MERGE CONGESTION INTO PL DATA
# ---------------------------------------------------------

pl_team_matches = (
    pl_team_matches
    .drop(
        columns=[
            "matches_last_7_days",
            "matches_last_14_days"
        ],
        errors="ignore"
    )
    .merge(
        congestion_df[
            [
                "match_id",
                "team",
                "matches_last_7_days",
                "matches_last_14_days"
            ]
        ],
        on=[
            "match_id",
            "team"
        ],
        how="left"
    )
)


# ---------------------------------------------------------
# CHECK 2026/27
# ---------------------------------------------------------

check_2026_congestion = (
    pl_team_matches[
        pl_team_matches["season"]
        == "2026_27"
    ]
    .sort_values(
        [
            "date",
            "team"
        ]
    )
)


print(
    "PL team matches shape:",
    pl_team_matches.shape
)

print(
    "2026/27 rows:",
    len(check_2026_congestion)
)

print()

print(
    "Missing matches_last_7_days:",
    check_2026_congestion[
        "matches_last_7_days"
    ].isna().sum()
)

print(
    "Missing matches_last_14_days:",
    check_2026_congestion[
        "matches_last_14_days"
    ].isna().sum()
)

print()

print(
    check_2026_congestion[
        [
            "date",
            "team",
            "opponent",
            "days_rest",
            "matches_last_7_days",
            "matches_last_14_days"
        ]
    ]
    .tail(12)
    .to_string(index=False)
)

In [ ]:
# ---------------------------------------------------------
# REBUILD LOCKED V1 CROSS-COMPETITION ELO
# ---------------------------------------------------------

import pandas as pd


STARTING_ELO = 1500
K_FACTOR = 20


# ---------------------------------------------------------
# SORT EVERY MATCH CHRONOLOGICALLY
# ---------------------------------------------------------

elo_source = (
    historical_df[
        [
            "match_id",
            "date",
            "home_team",
            "away_team",
            "home_goals",
            "away_goals"
        ]
    ]
    .dropna(
        subset=[
            "date",
            "home_goals",
            "away_goals"
        ]
    )
    .sort_values(
        [
            "date",
            "match_id"
        ]
    )
    .reset_index(drop=True)
)


ratings = {}
elo_rows = []


# ---------------------------------------------------------
# PROCESS MATCHES ONE AT A TIME
# ---------------------------------------------------------

for _, match in elo_source.iterrows():

    home_team = match["home_team"]
    away_team = match["away_team"]


    # First time we see a team -> 1500
    home_elo = ratings.get(
        home_team,
        STARTING_ELO
    )

    away_elo = ratings.get(
        away_team,
        STARTING_ELO
    )


    # -----------------------------------------------------
    # STORE PRE-MATCH ELO
    # -----------------------------------------------------

    elo_rows.append({

        "match_id":
            match["match_id"],

        "home_elo":
            home_elo,

        "away_elo":
            away_elo,

        "elo_difference":
            home_elo - away_elo
    })


    # -----------------------------------------------------
    # EXPECTED RESULT
    # NO HOME-ADVANTAGE BONUS
    # -----------------------------------------------------

    expected_home = (
        1
        /
        (
            1
            +
            10 ** (
                (away_elo - home_elo)
                / 400
            )
        )
    )


    expected_away = (
        1
        -
        expected_home
    )


    # -----------------------------------------------------
    # ACTUAL RESULT
    # WIN = 1
    # DRAW = 0.5
    # LOSS = 0
    # -----------------------------------------------------

    if (
        match["home_goals"]
        >
        match["away_goals"]
    ):

        actual_home = 1.0
        actual_away = 0.0


    elif (
        match["home_goals"]
        <
        match["away_goals"]
    ):

        actual_home = 0.0
        actual_away = 1.0


    else:

        actual_home = 0.5
        actual_away = 0.5


    # -----------------------------------------------------
    # UPDATE RATINGS AFTER THE MATCH
    # -----------------------------------------------------

    ratings[home_team] = (
        home_elo
        +
        K_FACTOR
        *
        (
            actual_home
            -
            expected_home
        )
    )


    ratings[away_team] = (
        away_elo
        +
        K_FACTOR
        *
        (
            actual_away
            -
            expected_away
        )
    )


# ---------------------------------------------------------
# CREATE ELO DATAFRAME
# ---------------------------------------------------------

elo_history_df = pd.DataFrame(
    elo_rows
)


print(
    "Elo history shape:",
    elo_history_df.shape
)

print(
    "Teams rated:",
    len(ratings)
)


# ---------------------------------------------------------
# MERGE PRE-MATCH ELO INTO PL TEAM DATA
# ---------------------------------------------------------

home_elo_df = (
    elo_history_df[
        [
            "match_id",
            "home_elo",
            "away_elo",
            "elo_difference"
        ]
    ]
    .rename(
        columns={
            "home_elo":
                "team_elo",

            "away_elo":
                "opponent_elo",

            "elo_difference":
                "team_elo_difference"
        }
    )
)


away_elo_df = (
    elo_history_df[
        [
            "match_id",
            "home_elo",
            "away_elo",
            "elo_difference"
        ]
    ]
    .rename(
        columns={
            "away_elo":
                "team_elo",

            "home_elo":
                "opponent_elo",

            "elo_difference":
                "home_elo_difference"
        }
    )
)


# Away team's own difference should be team - opponent
away_elo_df["team_elo_difference"] = (
    away_elo_df["team_elo"]
    -
    away_elo_df["opponent_elo"]
)


away_elo_df = away_elo_df[
    [
        "match_id",
        "team_elo",
        "opponent_elo",
        "team_elo_difference"
    ]
]


# ---------------------------------------------------------
# ADD ELO BASED ON VENUE
# ---------------------------------------------------------

pl_team_matches = (
    pl_team_matches
    .drop(
        columns=[
            "team_elo",
            "opponent_elo",
            "team_elo_difference"
        ],
        errors="ignore"
    )
)


pl_home = (
    pl_team_matches[
        pl_team_matches["venue"] == "home"
    ]
    .merge(
        home_elo_df,
        on="match_id",
        how="left"
    )
)


pl_away = (
    pl_team_matches[
        pl_team_matches["venue"] == "away"
    ]
    .merge(
        away_elo_df,
        on="match_id",
        how="left"
    )
)


pl_team_matches = pd.concat(
    [
        pl_home,
        pl_away
    ],
    ignore_index=True
)


pl_team_matches = (
    pl_team_matches
    .sort_values(
        [
            "team",
            "date",
            "match_id"
        ]
    )
    .reset_index(drop=True)
)


# ---------------------------------------------------------
# CHECK CURRENT SEASON
# ---------------------------------------------------------

check_2026_elo = (
    pl_team_matches[
        pl_team_matches["season"]
        == "2026_27"
    ]
    .sort_values(
        [
            "date",
            "team"
        ]
    )
)


print()

print(
    "PL team matches shape:",
    pl_team_matches.shape
)

print(
    "2026/27 rows:",
    len(check_2026_elo)
)

print()

print(
    "Missing team Elo:",
    check_2026_elo[
        "team_elo"
    ].isna().sum()
)

print(
    "Missing opponent Elo:",
    check_2026_elo[
        "opponent_elo"
    ].isna().sum()
)

print()

print(
    check_2026_elo[
        [
            "date",
            "team",
            "opponent",
            "venue",
            "team_elo",
            "opponent_elo",
            "team_elo_difference"
        ]
    ]
    .tail(12)
    .to_string(index=False)
)

In [ ]:
# ---------------------------------------------------------
# BUILD MATCH-LEVEL PREMIER LEAGUE FEATURE DATASET
# ---------------------------------------------------------

# Home team rows
home_match_features = (
    pl_team_matches[
        pl_team_matches["venue"] == "home"
    ]
    .copy()
)


# Away team rows
away_match_features = (
    pl_team_matches[
        pl_team_matches["venue"] == "away"
    ]
    .copy()
)


# ---------------------------------------------------------
# COLUMNS WE WANT TO KEEP
# ---------------------------------------------------------

feature_columns = [

    "points_last5_all",
    "goal_difference_last5_all",
    "win_rate_last5_all",
    "shots_on_target_difference_last5_all",

    "points_last5_pl",
    "goal_difference_last5_pl",
    "win_rate_last5_pl",
    "shots_on_target_difference_last5_pl",

    "points_last10_pl",
    "goal_difference_last10_pl",
    "win_rate_last10_pl",

    "days_rest",
    "matches_last_7_days",
    "matches_last_14_days",

    "team_elo"
]


# ---------------------------------------------------------
# HOME SIDE
# ---------------------------------------------------------

home_keep = (
    [
        "match_id",
        "date",
        "season",
        "team",
        "opponent",
        "goals_for",
        "goals_against"
    ]
    +
    feature_columns
)


home_match_features = (
    home_match_features[
        home_keep
    ]
    .rename(
        columns={
            "team":
                "home_team",

            "opponent":
                "away_team",

            "goals_for":
                "home_goals",

            "goals_against":
                "away_goals"
        }
    )
)


home_match_features = (
    home_match_features
    .rename(
        columns={
            col: f"{col}_home"
            for col in feature_columns
        }
    )
)


# ---------------------------------------------------------
# AWAY SIDE
# ---------------------------------------------------------

away_keep = (
    [
        "match_id",
        "team"
    ]
    +
    feature_columns
)


away_match_features = (
    away_match_features[
        away_keep
    ]
    .rename(
        columns={
            "team":
                "away_team_check"
        }
    )
)


away_match_features = (
    away_match_features
    .rename(
        columns={
            col: f"{col}_away"
            for col in feature_columns
        }
    )
)


# ---------------------------------------------------------
# MERGE HOME + AWAY
# ---------------------------------------------------------

model_df = (
    home_match_features
    .merge(
        away_match_features,
        on="match_id",
        how="left"
    )
)


# ---------------------------------------------------------
# VERIFY AWAY TEAM MATCH
# ---------------------------------------------------------

model_df["away_team_match"] = (
    model_df["away_team"]
    ==
    model_df["away_team_check"]
)


print(
    "Away team mismatches:",
    (~model_df["away_team_match"]).sum()
)


model_df = model_df.drop(
    columns=[
        "away_team_check",
        "away_team_match"
    ]
)


# ---------------------------------------------------------
# TARGET
# ---------------------------------------------------------

model_df["target"] = np.select(

    [
        model_df["home_goals"]
        >
        model_df["away_goals"],

        model_df["home_goals"]
        <
        model_df["away_goals"]
    ],

    [
        "H",
        "A"
    ],

    default="D"
)


# ---------------------------------------------------------
# LOCKED RF V1 ELO DIFFERENCE
# HOME ELO - AWAY ELO
# ---------------------------------------------------------

model_df["team_elo_difference_home"] = (
    model_df["team_elo_home"]
    -
    model_df["team_elo_away"]
)


# ---------------------------------------------------------
# LOCKED 31 FEATURES
# ---------------------------------------------------------

baseline_features = [

    "points_last5_all_home",
    "goal_difference_last5_all_home",
    "win_rate_last5_all_home",
    "shots_on_target_difference_last5_all_home",

    "points_last5_all_away",
    "goal_difference_last5_all_away",
    "win_rate_last5_all_away",
    "shots_on_target_difference_last5_all_away",

    "points_last5_pl_home",
    "goal_difference_last5_pl_home",
    "win_rate_last5_pl_home",
    "shots_on_target_difference_last5_pl_home",
    "points_last10_pl_home",
    "goal_difference_last10_pl_home",
    "win_rate_last10_pl_home",

    "points_last5_pl_away",
    "goal_difference_last5_pl_away",
    "win_rate_last5_pl_away",
    "shots_on_target_difference_last5_pl_away",
    "points_last10_pl_away",
    "goal_difference_last10_pl_away",
    "win_rate_last10_pl_away",

    "days_rest_home",
    "days_rest_away",
    "matches_last_7_days_home",
    "matches_last_7_days_away",
    "matches_last_14_days_home",
    "matches_last_14_days_away",

    "team_elo_home",
    "team_elo_away",
    "team_elo_difference_home"
]


# ---------------------------------------------------------
# CHECK
# ---------------------------------------------------------

model_2026 = (
    model_df[
        model_df["season"] == "2026_27"
    ]
    .sort_values("date")
)


print()

print(
    "Model dataframe shape:",
    model_df.shape
)

print(
    "Locked feature count:",
    len(baseline_features)
)

print(
    "2026/27 matches:",
    len(model_2026)
)

print()

print(
    "Missing locked-feature values in 2026/27:",
    model_2026[
        baseline_features
    ]
    .isna()
    .sum()
    .sum()
)

print()

print(
    model_2026[
        [
            "date",
            "home_team",
            "away_team",
            "home_goals",
            "away_goals",
            "target"
        ]
    ]
    .to_string(index=False)
)

In [ ]:
# ---------------------------------------------------------
# SAVE UPDATED MATCH-LEVEL FEATURE DATASET
# ---------------------------------------------------------

output_path = "../data/processed/pl_match_features.csv"


model_df.to_csv(
    output_path,
    index=False
)


print(
    "Saved:",
    output_path
)

print(
    "Shape:",
    model_df.shape
)

print(
    "Columns:",
    len(model_df.columns)
)

print()

print(
    "Seasons:"
)

print(
    model_df["season"]
    .value_counts()
    .sort_index()
)